<a href="https://colab.research.google.com/github/DenisKai7/herbal-pharma/blob/main/herbal_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import os
if 'COLAB_' not in ''.join(os.environ.keys()):
    !pip install unsloth sentence-transformers faiss-cpu rich colorama tabulate rouge_score bert_score
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 transformers==4.57.6 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf 'datasets>=3.4.1' huggingface_hub hf_transfer
    !pip install sentence-transformers faiss-cpu rich colorama tabulate
    !pip install rouge_score bert_score
    !pip install --no-deps unsloth
print('✅ Semua dependencies terinstall!')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 19.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.8/393.8 kB 11.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 64.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.3.1 requires msgspec, which is not installed.
unsloth-zoo 2026.3.1 requires t

In [2]:
sql_content = """
-- ============================================================
-- HERBALPHARMA-AGI DATABASE v1.0
-- Source of Truth: Farmakognosi & Fitokimia Herbal Indonesia
-- ============================================================

PRAGMA foreign_keys = ON;

-- ============================================================
-- TABLE: tanaman (Plants)
-- ============================================================
CREATE TABLE IF NOT EXISTS tanaman (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nama_ilmiah TEXT NOT NULL UNIQUE,
    nama_lokal TEXT,
    nama_inggris TEXT,
    familia TEXT,
    genus TEXT,
    spesies TEXT,
    deskripsi TEXT,
    bagian_digunakan TEXT,   -- akar, daun, buah, biji, kulit batang, rimpang
    habitat TEXT,
    negara_asal TEXT,
    status_bpom TEXT,        -- Terdaftar, Dalam Kajian, Tidak Terdaftar
    status_who TEXT,
    status_ema TEXT,
    catatan_keamanan TEXT,
    created_at TEXT DEFAULT (datetime('now'))
);

-- ============================================================
-- TABLE: senyawa (Compounds / Phytochemicals)
-- ============================================================
CREATE TABLE IF NOT EXISTS senyawa (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nama_senyawa TEXT NOT NULL,
    nama_iupac TEXT,
    cas_number TEXT,
    golongan TEXT,           -- alkaloid, flavonoid, terpenoid, tanin, saponin, dll
    rumus_molekul TEXT,
    berat_molekul REAL,
    kelarutan TEXT,
    bioavailabilitas TEXT,
    mekanisme_aksi TEXT,
    deskripsi TEXT
);

-- ============================================================
-- TABLE: tanaman_senyawa (Many-to-Many: Plant-Compound)
-- ============================================================
CREATE TABLE IF NOT EXISTS tanaman_senyawa (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    tanaman_id INTEGER NOT NULL,
    senyawa_id INTEGER NOT NULL,
    kadar_persentase REAL,
    bagian_tanaman TEXT,
    metode_ekstraksi TEXT,
    referensi TEXT,
    FOREIGN KEY (tanaman_id) REFERENCES tanaman(id),
    FOREIGN KEY (senyawa_id) REFERENCES senyawa(id)
);

-- ============================================================
-- TABLE: efek_farmakologi (Pharmacological Effects)
-- ============================================================
CREATE TABLE IF NOT EXISTS efek_farmakologi (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nama_efek TEXT NOT NULL UNIQUE,
    kategori TEXT,           -- antioksidan, antiinflamasi, antimikroba, dll
    mekanisme_umum TEXT,
    level_bukti TEXT,        -- A (kuat), B (sedang), C (lemah), D (in vitro saja)
    deskripsi TEXT
);

-- ============================================================
-- TABLE: tanaman_efek (Many-to-Many: Plant-Effect)
-- ============================================================
CREATE TABLE IF NOT EXISTS tanaman_efek (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    tanaman_id INTEGER NOT NULL,
    efek_id INTEGER NOT NULL,
    level_bukti TEXT,        -- A, B, C, D
    dosis_efektif TEXT,
    rute_pemberian TEXT,     -- oral, topikal, inhalasi
    populasi_studi TEXT,
    referensi TEXT,
    catatan TEXT,
    FOREIGN KEY (tanaman_id) REFERENCES tanaman(id),
    FOREIGN KEY (efek_id) REFERENCES efek_farmakologi(id)
);

-- ============================================================
-- TABLE: jalur_metabolit (Metabolic Pathways)
-- ============================================================
CREATE TABLE IF NOT EXISTS jalur_metabolit (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    senyawa_id INTEGER NOT NULL,
    enzim_cyp TEXT,          -- CYP3A4, CYP2D6, CYP2C9, dll
    tipe_interaksi TEXT,     -- inhibitor, inducer, substrat
    organ_target TEXT,
    metabolit_utama TEXT,
    waktu_paruh TEXT,
    protein_binding REAL,    -- persentase
    volume_distribusi TEXT,
    deskripsi TEXT,
    FOREIGN KEY (senyawa_id) REFERENCES senyawa(id)
);

-- ============================================================
-- TABLE: studi_klinis (Clinical Studies)
-- ============================================================
CREATE TABLE IF NOT EXISTS studi_klinis (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    tanaman_id INTEGER NOT NULL,
    judul_studi TEXT NOT NULL,
    tipe_studi TEXT,         -- RCT, Cohort, Case-Control, In Vitro, In Vivo, Meta-Analysis
    populasi TEXT,
    ukuran_sampel INTEGER,
    durasi_studi TEXT,
    dosis_uji TEXT,
    endpoint_primer TEXT,
    hasil TEXT,
    kesimpulan TEXT,
    level_evidens TEXT,      -- I, II, III, IV, V
    jurnal TEXT,
    tahun INTEGER,
    doi TEXT,
    FOREIGN KEY (tanaman_id) REFERENCES tanaman(id)
);

-- ============================================================
-- TABLE: interaksi_obat (Drug Interactions)
-- ============================================================
CREATE TABLE IF NOT EXISTS interaksi_obat (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    tanaman_id INTEGER NOT NULL,
    obat_konvensional TEXT NOT NULL,
    tipe_interaksi TEXT,     -- farmakokinetik, farmakodinamik
    mekanisme TEXT,
    tingkat_keparahan TEXT,  -- Minor, Moderate, Major, Contraindicated
    efek_interaksi TEXT,
    rekomendasi_klinis TEXT,
    referensi TEXT,
    FOREIGN KEY (tanaman_id) REFERENCES tanaman(id)
);

-- ============================================================
-- TABLE: keamanan_toksikologi (Safety & Toxicology)
-- ============================================================
CREATE TABLE IF NOT EXISTS keamanan_toksikologi (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    tanaman_id INTEGER NOT NULL,
    ld50 TEXT,               -- mg/kg
    noael TEXT,
    kategori_kehamilan TEXT, -- A, B, C, D, X
    aman_laktasi INTEGER,    -- 0=tidak, 1=ya, 2=data tidak cukup
    aman_anak INTEGER,
    kontraindikasi TEXT,
    efek_samping TEXT,
    dosis_maksimal_harian TEXT,
    dosis_terapi_umum TEXT,
    toksisitas_organ TEXT,
    catatan_khusus TEXT,
    FOREIGN KEY (tanaman_id) REFERENCES tanaman(id)
);

-- ============================================================
-- TABLE: regulasi (Regulatory Information)
-- ============================================================
CREATE TABLE IF NOT EXISTS regulasi (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    tanaman_id INTEGER NOT NULL,
    lembaga TEXT NOT NULL,   -- BPOM, WHO, EMA, FDA
    status TEXT,             -- Approved, Under Review, Banned, Traditional Use Only
    nomor_registrasi TEXT,
    kategori_produk TEXT,    -- OHT, Fitofarmaka, Jamu, Suplemen
    klaim_disetujui TEXT,
    klaim_dilarang TEXT,
    batas_dosis_regulasi TEXT,
    tahun_regulasi INTEGER,
    dokumen_referensi TEXT,
    catatan TEXT,
    FOREIGN KEY (tanaman_id) REFERENCES tanaman(id)
);

-- ============================================================
-- TABLE: cara_pengolahan (Processing Methods)
-- ============================================================
CREATE TABLE IF NOT EXISTS cara_pengolahan (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    tanaman_id INTEGER NOT NULL,
    metode TEXT,             -- rebusan, seduhan, ekstrak, kapsul, tincture
    bagian_digunakan TEXT,
    takaran_bahan TEXT,
    volume_pelarut TEXT,
    suhu_proses TEXT,
    durasi_proses TEXT,
    cara_konsumsi TEXT,
    frekuensi TEXT,
    catatan_keamanan TEXT,
    level_pengguna TEXT,     -- umum, medis
    FOREIGN KEY (tanaman_id) REFERENCES tanaman(id)
);

-- ============================================================
-- TABLE: audit_log (Audit Trail)
-- ============================================================
CREATE TABLE IF NOT EXISTS audit_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT,
    user_role TEXT,
    query_text TEXT,
    intent_detected TEXT,
    sql_executed TEXT,
    response_type TEXT,      -- answered, rejected, partial
    gatekeeper_flag TEXT,
    timestamp TEXT DEFAULT (datetime('now'))
);

-- ============================================================
-- TABLE: feedback_log (User Feedback)
-- ============================================================
CREATE TABLE IF NOT EXISTS feedback_log (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    session_id TEXT,
    query_text TEXT,
    response_text TEXT,
    rating INTEGER,          -- 1-5
    feedback_text TEXT,
    user_role TEXT,
    timestamp TEXT DEFAULT (datetime('now'))
);

-- ============================================================
-- SEED DATA: TANAMAN
-- ============================================================
INSERT INTO tanaman (nama_ilmiah, nama_lokal, nama_inggris, familia, genus, spesies, deskripsi, bagian_digunakan, habitat, negara_asal, status_bpom, status_who, catatan_keamanan) VALUES

('Curcuma longa', 'Kunyit', 'Turmeric', 'Zingiberaceae', 'Curcuma', 'longa',
 'Tanaman rimpang tropis yang telah digunakan dalam pengobatan tradisional selama ribuan tahun. Mengandung kurkuminoid sebagai senyawa bioaktif utama.',
 'Rimpang', 'Tropis, tanah lembab', 'India, Asia Tenggara', 'Terdaftar', 'Traditional Medicine', 'Aman pada dosis terapi; hindari dosis tinggi pada penderita batu empedu'),

('Zingiber officinale', 'Jahe', 'Ginger', 'Zingiberaceae', 'Zingiber', 'officinale',
 'Tanaman rimpang yang dikenal sebagai bumbu dan obat tradisional. Gingerol dan shogaol merupakan senyawa aktif utama yang bertanggung jawab atas efek farmakologi.',
 'Rimpang', 'Tropis, tanah gembur', 'Asia Selatan', 'Terdaftar', 'Traditional Medicine',
 'Aman untuk kebanyakan orang; hati-hati pada gangguan pembekuan darah'),

('Andrographis paniculata', 'Sambiloto', 'King of Bitters', 'Acanthaceae', 'Andrographis', 'paniculata',
 'Tanaman herbal yang dikenal memiliki rasa sangat pahit. Andrografolid adalah senyawa diterpen aktif utama dengan aktivitas imunomodulator dan antiinflamasi yang terdokumentasi baik.',
 'Daun, Seluruh tanaman', 'Tropis, Asia Selatan', 'India, Asia Tenggara', 'Terdaftar', 'Traditional Medicine',
 'Hindari pada kehamilan; dapat menurunkan tekanan darah'),

('Morinda citrifolia', 'Mengkudu', 'Noni', 'Rubiaceae', 'Morinda', 'citrifolia',
 'Buah tropis yang digunakan dalam pengobatan tradisional Polinesia. Mengandung iridoid glikosida, antrakuinon, dan berbagai senyawa fenolik.',
 'Buah, Daun, Akar', 'Tropis, pantai', 'Polinesia, Asia Tenggara', 'Terdaftar', 'Traditional Medicine',
 'Tinggi kalium, hindari pada gagal ginjal; potensi hepatotoksisitas pada dosis tinggi'),

('Centella asiatica', 'Pegagan', 'Gotu Kola', 'Apiaceae', 'Centella', 'asiatica',
 'Tanaman herbal yang banyak digunakan dalam pengobatan Ayurveda dan tradisional Asia. Asiatikosida dan madekasosida dikenal mendukung penyembuhan luka dan fungsi kognitif.',
 'Seluruh tanaman, Daun', 'Tropis, lahan basah', 'Asia Tropis', 'Terdaftar', 'Traditional Medicine',
 'Aman pada dosis terapi; kasus hepatotoksisitas jarang dilaporkan pada penggunaan jangka panjang'),

('Kaempferia galanga', 'Kencur', 'Aromatic Ginger', 'Zingiberaceae', 'Kaempferia', 'galanga',
 'Tanaman rimpang aromatik yang banyak digunakan dalam masakan dan pengobatan tradisional Indonesia. Etil p-metoksisinamat adalah senyawa bioaktif utama.',
 'Rimpang', 'Tropis, Asia Tenggara', 'Indonesia, Malaysia', 'Terdaftar', 'Traditional Medicine',
 'Aman pada dosis kuliner; data toksikologi sistemik terbatas'),

('Nigella sativa', 'Jintan Hitam', 'Black Seed', 'Ranunculaceae', 'Nigella', 'sativa',
 'Tanaman annual yang bijinya telah digunakan selama berabad-abad dalam pengobatan Islam tradisional. Timokuinon adalah senyawa aktif utama dengan aktivitas antiinflamasi dan imunomodulator.',
 'Biji, Minyak biji', 'Semi-arid, Mediterania', 'Timur Tengah', 'Terdaftar', 'Traditional Medicine',
 'Aman pada dosis terapi; hindari dosis tinggi minyak pada kehamilan'),

('Orthosiphon aristatus', 'Kumis Kucing', 'Cat Whiskers', 'Lamiaceae', 'Orthosiphon', 'aristatus',
 'Tanaman asli Asia Tenggara yang dikenal sebagai diuretik herbal. Sinensetin dan rosmarinat adalah senyawa aktif utama dengan efek diuretik dan antiinflamasi.',
 'Daun', 'Tropis, Asia Tenggara', 'Indonesia, Malaysia', 'Terdaftar', 'Traditional Medicine',
 'Diuretik kuat, monitor elektrolit pada penggunaan jangka panjang'),

('Phylanthus niruri', 'Meniran', 'Stonebreaker', 'Phyllanthaceae', 'Phyllanthus', 'niruri',
 'Tanaman herbal yang digunakan dalam pengobatan tradisional Ayurveda dan Asia Tenggara. Filantin dan hipofilantin merupakan senyawa aktif dengan efek hepatoprotektif dan imunomodulator.',
 'Seluruh tanaman', 'Tropis, lahan terbuka', 'Asia Tropis', 'Terdaftar', 'Traditional Medicine',
 'Dapat menurunkan tekanan darah; hati-hati pada hipoglikemik'),

('Psidium guajava', 'Jambu Biji', 'Guava', 'Myrtaceae', 'Psidium', 'guajava',
 'Pohon buah tropis yang daunnya banyak digunakan untuk pengobatan diare dan diabetes. Quersetin dan asam elagat adalah senyawa bioaktif utama.',
 'Daun, Buah, Kulit batang', 'Tropis', 'Amerika Tropis', 'Terdaftar', 'Traditional Medicine',
 'Aman untuk konsumsi umum; penggunaan daun dalam dosis tinggi perlu pengawasan');

-- ============================================================
-- SEED DATA: SENYAWA
-- ============================================================
INSERT INTO senyawa (nama_senyawa, nama_iupac, cas_number, golongan, rumus_molekul, berat_molekul, kelarutan, bioavailabilitas, mekanisme_aksi, deskripsi) VALUES

('Kurkumin', '(1E,6E)-1,7-bis(4-hydroxy-3-methoxyphenyl)hepta-1,6-diene-3,5-dione',
 '458-37-7', 'Kurkuminoid', 'C21H20O6', 368.38,
 'Larut dalam aseton, etanol; tidak larut air',
 'Rendah (oral < 1%), meningkat dengan piperin',
 'Inhibisi NF-κB, COX-2, LOX; aktivasi Nrf2; modulasi sitokin proinflamasi',
 'Pigmen kuning utama kunyit; memiliki aktivitas antiinflamasi, antioksidan, dan antikanker yang luas'),

('Gingerol', '(S)-5-Hydroxy-1-(4-hydroxy-3-methoxyphenyl)decan-3-one',
 '23513-14-6', 'Fenilalkilketon', 'C17H26O4', 294.38,
 'Larut dalam etanol, kloroform',
 'Sedang, metabolisme first-pass',
 'Inhibisi COX-1/2; aktivasi TRPV1; inhibisi serotonin reseptor 5-HT3',
 'Senyawa pedas aktif jahe segar; antiemetik, antiinflamasi, dan analgesik'),

('Shogaol', '(E)-1-(4-Hydroxy-3-methoxyphenyl)dec-4-en-3-one',
 '555-66-8', 'Fenilalkilketon', 'C17H24O3', 276.37,
 'Larut dalam etanol',
 'Lebih tinggi dari gingerol',
 'Inhibisi COX-2; aktivasi TRPV1; modulasi TNF-α dan IL-6',
 'Terbentuk dari gingerol saat pemanasan; lebih poten sebagai antiinflamasi'),

('Andrografolid', 'Andrographolide', '5508-58-7', 'Diterpen Lakton', 'C20H30O5', 350.45,
 'Larut dalam metanol, etanol; sedikit larut air',
 'Cepat terabsorpsi, eliminasi cepat',
 'Inhibisi NF-κB; aktivasi Nrf2; inhibisi replikasi virus; stimulasi imun',
 'Senyawa pahit utama sambiloto; imunomodulator dan antiviral terdokumentasi'),

('Asiatikosida', 'Asiaticoside', '16830-15-2', 'Triterpen Glikosida', 'C48H78O19', 959.12,
 'Larut dalam air dan etanol',
 'Dihidrolisis menjadi asiatik acid aktif',
 'Stimulasi sintesis kolagen tipe I; aktivasi fibroblast; neuroproteksi via BDNF',
 'Senyawa utama pegagan; mendukung penyembuhan luka dan fungsi kognitif'),

('Timokuinon', 'Thymoquinone (2-isopropyl-5-methylbenzo-1,4-quinone)',
 '490-91-5', 'Kuinon', 'C10H12O2', 164.20,
 'Larut dalam etanol, aseton; sedikit larut air',
 'Cukup baik, metabolisme hepatik',
 'Inhibisi NF-κB dan COX; aktivasi apoptosis; modulasi sitokin',
 'Senyawa aktif utama jintan hitam; antiinflamasi, antioksidan, antikanker'),

('Sinensetin', 'Sinensetin', '2306-27-6', 'Polimetoksi Flavon', 'C20H20O7', 372.37,
 'Larut dalam etanol dan DMSO',
 'Sedang',
 'Inhibisi fosfodiesterase; efek diuretik; antiinflamasi',
 'Flavon utama daun kumis kucing; bertanggung jawab atas efek diuretik'),

('Filantin', 'Phyllanthin', '10041-05-1', 'Lignan', 'C24H34O6', 418.52,
 'Larut dalam metanol dan kloroform',
 'Sedang',
 'Hepatoprotektif; inhibisi replikasi HBV; imunomodulasi',
 'Lignan utama meniran; hepatoprotektif dan antiviral'),

('Quersetin', 'Quercetin (3,3'',4'',5,7-Pentahydroxyflavone)',
 '117-39-5', 'Flavonol', 'C15H10O7', 302.24,
 'Sedikit larut air, larut etanol',
 'Rendah-sedang, meningkat dengan makanan',
 'Inhibisi phosphoinositide 3-kinase; stabilisasi membran sel mast; antioksidan radikal bebas',
 'Flavonoid paling umum; ditemukan di berbagai tanaman termasuk jambu biji'),

('Etil p-Metoksisinamat', 'Ethyl p-methoxycinnamate', '24393-56-4', 'Sinamat Ester', 'C12H14O3', 206.24,
 'Larut dalam etanol dan kloroform',
 'Data terbatas',
 'Antiinflamasi via inhibisi COX; analgesik via jalur sentral',
 'Senyawa aktif utama kencur; antiinflamasi dan antijamur');

-- ============================================================
-- SEED DATA: TANAMAN-SENYAWA MAPPING
-- ============================================================
INSERT INTO tanaman_senyawa (tanaman_id, senyawa_id, kadar_persentase, bagian_tanaman, metode_ekstraksi, referensi) VALUES
(1, 1, 3.14, 'Rimpang', 'Ekstraksi etanol 70%', 'Aggarwal et al., 2007, J Biol Chem'),
(2, 2, 0.5, 'Rimpang segar', 'Ekstraksi air panas', 'Jolad et al., 2005, Phytochemistry'),
(2, 3, 0.4, 'Rimpang kering', 'Ekstraksi etanol', 'Jolad et al., 2005, Phytochemistry'),
(3, 4, 1.8, 'Daun', 'Ekstraksi metanol', 'Akbar et al., 2011, J Ethnopharmacol'),
(5, 5, 1.1, 'Seluruh tanaman', 'Ekstraksi etanol 70%', 'James & Dubery, 2009, Molecules'),
(7, 6, 1.4, 'Biji', 'Cold press / Ekstraksi etanol', 'Randhawa & Alghamdi, 2011, JEAS'),
(8, 7, 0.6, 'Daun', 'Ekstraksi etanol 80%', 'Yam et al., 2007, J Ethnopharmacol'),
(9, 8, 0.8, 'Seluruh tanaman', 'Ekstraksi metanol', 'Calixto et al., 1998, Planta Med'),
(10, 9, 0.3, 'Daun', 'Ekstraksi etanol', 'Morais-Braga et al., 2016, Molecules'),
(6, 10, 2.1, 'Rimpang', 'Ekstraksi etanol 95%', 'Wulandari et al., 2018, Pharmacogn J');

-- ============================================================
-- SEED DATA: EFEK FARMAKOLOGI
-- ============================================================
INSERT INTO efek_farmakologi (nama_efek, kategori, mekanisme_umum, level_bukti, deskripsi) VALUES
('Antiinflamasi', 'Anti-inflamasi', 'Inhibisi COX-1/2, LOX, NF-κB; penurunan sitokin proinflamasi (TNF-α, IL-1β, IL-6)', 'A', 'Mengurangi proses inflamasi akut dan kronik'),
('Antioksidan', 'Antioksidan', 'Scavenging radikal bebas ROS/RNS; aktivasi enzim endogen (SOD, CAT, GPx); chelasi logam', 'A', 'Melindungi sel dari stres oksidatif'),
('Antimikroba', 'Antimikroba', 'Kerusakan membran sel bakteri; inhibisi sintesis DNA/protein; gangguan biofilm', 'B', 'Aktivitas terhadap bakteri gram positif dan negatif'),
('Antidiabetes', 'Metabolisme', 'Inhibisi alfa-glukosidase dan alfa-amilase; peningkatan sekresi insulin; aktivasi AMPK', 'B', 'Membantu regulasi kadar gula darah'),
('Hepatoprotektif', 'Hepatologi', 'Penurunan ALT/AST; antioksidan hepatosit; antiinflamasi hati; regenerasi hepatosit', 'B', 'Melindungi sel hati dari kerusakan'),
('Imunomodulator', 'Imunologi', 'Stimulasi atau regulasi sel imun (NK, T-cell, makrofag); modulasi sitokin', 'B', 'Memodulasi respons imun tubuh'),
('Antikanker', 'Onkologi', 'Induksi apoptosis; inhibisi proliferasi sel; anti-angiogenesis; inhibisi metastasis', 'C', 'Aktivitas antitumor, umumnya masih in vitro/in vivo'),
('Antihipertensi', 'Kardiovaskuler', 'Inhibisi ACE; vasodilatasi; diuretik ringan; penurunan resistensi perifer', 'B', 'Membantu menurunkan tekanan darah'),
('Diuretik', 'Ginjal', 'Peningkatan ekskresi Na+/K+ renal; inhibisi reabsorpsi tubular', 'B', 'Meningkatkan produksi dan ekskresi urin'),
('Neuroprotektif', 'Neurologi', 'Peningkatan BDNF/NGF; perlindungan mitokondria; pengurangan stres oksidatif otak', 'C', 'Melindungi sel saraf dari kerusakan'),
('Antiemetik', 'Gastroenterologi', 'Inhibisi reseptor 5-HT3; antagonis dopamin; modulasi motilitas GI', 'A', 'Mengurangi mual dan muntah'),
('Penyembuhan Luka', 'Dermatologi', 'Stimulasi sintesis kolagen; proliferasi fibroblast; angiogenesis; re-epitelialisasi', 'B', 'Mempercepat proses penyembuhan luka');

-- ============================================================
-- SEED DATA: TANAMAN-EFEK MAPPING
-- ============================================================
INSERT INTO tanaman_efek (tanaman_id, efek_id, level_bukti, dosis_efektif, rute_pemberian, populasi_studi, referensi, catatan) VALUES
(1, 1, 'A', '500-1000 mg kurkumin/hari', 'Oral', 'Manusia (RCT)', 'Hewlings & Kalman, 2017, Foods', 'Bioavailabilitas meningkat dengan piperin 20mg'),
(1, 2, 'A', '400-600 mg ekstrak/hari', 'Oral', 'In vitro, In vivo', 'Menon & Sudheer, 2007, AEMB', NULL),
(1, 7, 'C', '500-2000 mg/hari', 'Oral', 'In vitro, beberapa uji klinis awal', 'Kunnumakkara et al., 2017, J Exp Clin Cancer Res', 'Bukti klinis masih terbatas'),
(2, 11, 'A', '1 g jahe/hari', 'Oral', 'Manusia (RCT - mual kehamilan)', 'Viljoen et al., 2014, Nutr J', 'Aman untuk kehamilan trimester 1 sesuai dosis'),
(2, 1, 'B', '2 g jahe segar/hari', 'Oral', 'Manusia, Hewan', 'Mashhadi et al., 2013, Int J Prev Med', NULL),
(3, 6, 'B', '200 mg andrografolid/hari', 'Oral', 'Manusia, RCT', 'Coon & Ernst, 2004, Planta Med', 'Efektif pada infeksi saluran napas atas'),
(3, 1, 'B', '400 mg ekstrak/hari', 'Oral', 'In vitro, In vivo', 'Gupta et al., 2017, J Ethnopharmacol', NULL),
(4, 2, 'B', 'Jus buah 500 ml/hari', 'Oral', 'Manusia', 'Wang et al., 2002, Acta Pharmacol Sin', NULL),
(5, 12, 'B', 'Krim topikal 1% asiatikosida', 'Topikal', 'Manusia, RCT', 'Bylka et al., 2014, Adv Dermatol Allergol', 'Efektif untuk luka kronik dan bekas luka'),
(5, 10, 'C', '60 mg ekstrak standar/hari', 'Oral', 'Manusia, beberapa RCT', 'Dev et al., 2009, IJBS', 'Perlu lebih banyak RCT berkualitas tinggi'),
(7, 6, 'B', '300-500 mg ekstrak/hari', 'Oral', 'Manusia, RCT', 'Salem, 2005, Phytomedicine', NULL),
(7, 1, 'B', '2-5 ml minyak/hari', 'Oral', 'Manusia, Hewan', 'Yimer et al., 2019, Evid Based Complement Alternat Med', NULL),
(8, 9, 'B', '3 g daun kering/hari (infus)', 'Oral', 'Manusia, uji klinis terbatas', 'Schut & Zwaving, 1993, Pharm Weekbl', 'Monitor kalium pada penggunaan > 4 minggu'),
(9, 5, 'B', '900 mg ekstrak/hari', 'Oral', 'In vitro, In vivo', 'Canigueral et al., 1995, Planta Med', NULL),
(9, 6, 'B', '600-900 mg/hari', 'Oral', 'In vivo, In vitro', 'Thyagarajan et al., 1990, Lancet', NULL),
(10, 4, 'B', 'Rebusan 15-30 g daun/hari', 'Oral', 'Manusia, uji klinis awal', 'Bhatia & Sharma, 2014, Phytother Res', NULL),
(10, 3, 'B', 'Ekstrak daun 250-500 mg/hari', 'Oral', 'In vitro', 'Morais-Braga et al., 2016, Molecules', 'Aktivitas terhadap S. aureus dan E. coli');

-- ============================================================
-- SEED DATA: JALUR METABOLIT
-- ============================================================
INSERT INTO jalur_metabolit (senyawa_id, enzim_cyp, tipe_interaksi, organ_target, metabolit_utama, waktu_paruh, protein_binding, volume_distribusi, deskripsi) VALUES
(1, 'CYP1A2, CYP3A4', 'Inhibitor lemah', 'Hati, Usus', 'Tetrahidrokurukumin, Heksahidrokurukumin', '6-8 jam', 65.0, 'Tinggi (distribusi ke jaringan)', 'Dimetabolisme ekstensif di hati; konjugasi glukoronidasi dan sulfasi'),
(2, 'CYP3A4', 'Substrat', 'Hati', 'Gingerol-glukoronida', '4-6 jam', 45.0, 'Sedang', 'Metabolisme fase II via glukoronidasi; eliminasi ginjal dan bilier'),
(4, 'CYP3A4, CYP2C9', 'Inhibitor sedang', 'Hati, Ginjal', 'Andrografolid-19β-D-glukoronida', '3-5 jam', 35.0, 'Luas', 'Cepat terabsorpsi; eliminasi terutama melalui empedu'),
(6, 'CYP2B6, CYP2C19', 'Inhibitor lemah-sedang', 'Hati', 'Timohidrokuinon', '8-12 jam', 50.0, 'Sedang', 'Metabolisme oksidatif; aktivitas metabolit lebih rendah dari senyawa induk'),
(9, 'CYP3A4, CYP1A1', 'Inhibitor lemah', 'Hati, Usus', 'Quersetin-3-glukoronida, Isorhamnetin', '11-28 jam', 76.0, 'Luas', 'Terkonjugasi di usus dan hati; metabolit aktif biologis');

-- ============================================================
-- SEED DATA: STUDI KLINIS
-- ============================================================
INSERT INTO studi_klinis (tanaman_id, judul_studi, tipe_studi, populasi, ukuran_sampel, durasi_studi, dosis_uji, endpoint_primer, hasil, kesimpulan, level_evidens, jurnal, tahun, doi) VALUES

(1, 'Efficacy of Curcumin in the Management of Chronic Anterior Uveitis',
 'RCT', 'Pasien uveitis anterior kronik', 32, '12 minggu', '375 mg kurkumin 3x/hari',
 'Inflamasi okuler (slit lamp score)', 'Perbaikan signifikan pada kelompok kurkumin vs kontrol (p<0.01)',
 'Kurkumin efektif dan aman untuk uveitis anterior kronik', 'II', 'Phytotherapy Research', 1999,
 '10.1002/(SICI)1099-1573(199903)13:2<123::AID-PTR399>3.0.CO;2-B'),

(2, 'Ginger for Nausea and Vomiting in Pregnancy: Randomized Double-Masked, Placebo-Controlled Trial',
 'RCT', 'Wanita hamil < 17 minggu', 170, '4 minggu', '250 mg ekstrak jahe 4x/hari',
 'Skor mual (PUQE score)', 'Reduksi mual signifikan pada kelompok jahe vs plasebo (p<0.001)',
 'Jahe aman dan efektif untuk mual kehamilan trimester 1-2', 'I', 'Obstetrics & Gynecology', 2005,
 '10.1097/01.AOG.0000172397.38084.62'),

(3, 'Efficacy and Safety of Andrographis paniculata Extract in Patients with Uncomplicated Upper Respiratory Tract Infection',
 'RCT', 'Pasien dewasa dengan ISPA ringan', 107, '5 hari', '1200 mg EP80 andrografis/hari',
 'Skor gejala ISPA (panas, sakit tenggorokan, lemas)', 'Pengurangan signifikan semua gejala pada hari ke-5 (p<0.05)',
 'Andrographis efektif mempercepat pemulihan ISPA ringan', 'II', 'Phytomedicine', 2009,
 '10.1016/j.phymed.2008.09.007'),

(5, 'Effect of Centella asiatica on Mild Cognitive Impairment: A Randomized Controlled Trial',
 'RCT', 'Lansia dengan gangguan kognitif ringan', 60, '6 minggu', '750 mg ekstrak/hari',
 'Skor MMSE, Digit Span, Trail Making Test', 'Peningkatan memori kerja dan fungsi eksekutif (p<0.05)',
 'Pegagan berpotensi mendukung fungsi kognitif pada lansia', 'II', 'Journal of Ethnopharmacology', 2016,
 '10.1016/j.jep.2016.02.029'),

(9, 'Clinical Trial of Phyllanthus niruri in Chronic Hepatitis B',
 'RCT', 'Pasien Hepatitis B kronik', 59, '30 hari', '900 mg ekstrak standar/hari',
 'Klirens HBsAg, normalisasi ALT', 'Klirens HBsAg pada 59% kelompok meniran vs 4% plasebo',
 'Meniran berpotensi sebagai antiviral HBV, perlu studi lebih besar', 'II', 'Lancet', 1988,
 '10.1016/S0140-6736(88)92111-3');

-- ============================================================
-- SEED DATA: INTERAKSI OBAT
-- ============================================================
INSERT INTO interaksi_obat (tanaman_id, obat_konvensional, tipe_interaksi, mekanisme, tingkat_keparahan, efek_interaksi, rekomendasi_klinis, referensi) VALUES
(1, 'Warfarin', 'Farmakokinetik', 'Inhibisi CYP3A4 meningkatkan kadar warfarin; efek antiplatelet aditif', 'Moderate', 'Peningkatan risiko perdarahan, perpanjangan INR', 'Monitor INR ketat; informasikan dokter jika menggunakan suplemen kunyit', 'Gorski et al., 2004, Clin Pharmacol Ther'),
(1, 'Antiplatelet (Aspirin, Klopidogrel)', 'Farmakodinamik', 'Efek antiplatelet aditif kurkumin dan obat antiplatelet', 'Moderate', 'Peningkatan risiko perdarahan', 'Gunakan dengan hati-hati; hindari dosis tinggi bersamaan', 'Ramirez-Bosca et al., 2000, Age'),
(2, 'Antikoagulan', 'Farmakodinamik', 'Gingerol menghambat aggregasi platelet via TXA2', 'Minor-Moderate', 'Potensi peningkatan perdarahan', 'Monitor jika dosis tinggi (>4g/hari); dosis kuliner umumnya aman', 'Lumb, 2002, Anaesthesia'),
(3, 'Antihipertensi', 'Farmakodinamik', 'Andrografolid memiliki efek hipotensif; efek aditif', 'Moderate', 'Hipotensi berlebihan', 'Monitor tekanan darah secara rutin', 'Burgos et al., 2009, J Ethnopharmacol'),
(7, 'Siklosporin', 'Farmakokinetik', 'Timokuinon dapat inhibisi CYP3A4 dan P-glikoprotein', 'Major', 'Peningkatan kadar siklosporin, risiko nefrotoksisitas', 'HINDARI kombinasi atau monitor ketat kadar siklosporin dalam darah', 'Ghlissi et al., 2014, Biomed Pharmacother'),
(8, 'Diuretik (Furosemid, Hidroklorotiazid)', 'Farmakodinamik', 'Efek diuretik aditif kumis kucing dan diuretik farmasi', 'Moderate', 'Risiko dehidrasi dan hipokalemia', 'Monitor elektrolit secara berkala; batasi dosis kumis kucing', 'Schut & Zwaving, 1993, Pharm Weekbl'),
(4, 'Obat Hepatotoksik (Paracetamol dosis tinggi)', 'Farmakodinamik', 'Komponen mengkudu dilaporkan memiliki potensi hepatotoksisitas', 'Moderate', 'Risiko kerusakan hati meningkat', 'Hindari kombinasi; monitor fungsi hati (ALT, AST)', 'Stadlbauer et al., 2005, J Hepatol');

-- ============================================================
-- SEED DATA: KEAMANAN TOKSIKOLOGI
-- ============================================================
INSERT INTO keamanan_toksikologi (tanaman_id, ld50, noael, kategori_kehamilan, aman_laktasi, aman_anak, kontraindikasi, efek_samping, dosis_maksimal_harian, dosis_terapi_umum, toksisitas_organ, catatan_khusus) VALUES
(1, '> 2000 mg/kg (tikus, oral)', '100 mg/kg/hari', 'B', 1, 1,
 'Obstruksi saluran empedu; batu empedu simptomatik; alergi Zingiberaceae',
 'GI ringan (mual, diare) pada dosis tinggi; pewarnaan tinja kuning',
 '8 gram kunyit bubuk/hari (setara ~500mg kurkumin terstandar)',
 '1-3 gram bubuk rimpang/hari; 400-600 mg ekstrak terstandar',
 'Hepatotoksisitas sangat jarang, dilaporkan pada dosis sangat tinggi',
 'Formulasi dengan piperin meningkatkan bioavailabilitas 20x'),

(2, '> 5000 mg/kg (tikus, oral)', '250 mg/kg/hari', 'A', 1, 1,
 'Batu empedu; gangguan pembekuan darah; sebelum operasi besar (hentikan 2 minggu sebelum)',
 'Heartburn; GERD pada dosis tinggi',
 '4 gram jahe kering/hari (umum); 1 gram/hari untuk mual kehamilan',
 '0.5-1 gram jahe kering 3x/hari; 250 mg ekstrak standar 4x/hari',
 'Tidak ada hepatotoksisitas terlaporkan pada dosis terapi',
 'Jahe segar lebih aman dari ekstrak pekat untuk penggunaan umum'),

(3, '> 2000 mg/kg (tikus)', '60 mg/kg/hari', 'X', 0, 2,
 'KEHAMILAN (dapat menyebabkan keguguran - uterotonik); hipersensitivitas; gangguan imunologi berat',
 'Penurunan tekanan darah; mual; alergi kulit; penurunan fertilitas (pada studi hewan)',
 '6 mg andrografolid/hari',
 '3-6 mg andrografolid/hari; 400 mg ekstrak standar 3x/hari',
 'Hepatotoksisitas dilaporkan pada penggunaan jangka sangat panjang',
 'KONTRAINDIKASI MUTLAK PADA KEHAMILAN'),

(5, '> 1000 mg/kg (tikus)', '30 mg/kg/hari', 'C', 2, 2,
 'Hipersensitivitas; penyakit hati berat',
 'Sakit kepala; pusing; hepatotoksisitas (kasus jarang)',
 '60-180 mg ekstrak standar/hari',
 '30-60 mg ekstrak standar (20-40% asiatikosida) 2-3x/hari',
 'Beberapa laporan kasus hepatotoksisitas idiosinkratik',
 'Monitor fungsi hati pada penggunaan > 6 minggu'),

(7, '> 2000 mg/kg minyak (tikus)', '100 mg/kg/hari', 'C', 2, 2,
 'Transplantasi organ (interaksi siklosporin); hipersensitivitas',
 'GI ringan; ruam kulit; penurunan jumlah sperma (pada studi hewan dosis tinggi)',
 '5 ml minyak/hari atau 3 gram biji/hari',
 '0.5-2 ml minyak/hari; 1-3 gram biji bubuk/hari',
 'Nefrotoksisitas pada dosis sangat tinggi (studi hewan)',
 'Hati-hati pada pasien transplantasi organ'),

(8, '> 5000 mg/kg (tikus)', '200 mg/kg/hari', 'C', 2, 2,
 'Hipokalemia; gagal ginjal berat; kehamilan (data terbatas)',
 'Hipokalemia pada penggunaan berlebihan; diuresis berlebihan',
 '3 gram daun kering/hari (infus)',
 '1-2 gram daun kering/infus 2x/hari',
 'Toksisitas ginjal pada dosis sangat tinggi (studi hewan)',
 'Monitor kalium darah pada penggunaan > 4 minggu'),

(4, 'Data tidak cukup untuk LD50 definitif pada manusia', 'Data terbatas', 'C', 2, 2,
 'Penyakit ginjal berat (tinggi kalium); penyakit hati; kehamilan (data terbatas)',
 'Hepatotoksisitas (beberapa laporan kasus); diare; mual',
 '750 ml jus buah/hari',
 '30-90 ml jus/hari; 500 mg ekstrak standar/hari',
 'HEPATOTOKSISITAS: beberapa laporan kasus terdokumentasi di literatur',
 'Tidak dianjurkan untuk pasien penyakit hati atau ginjal berat'),

(9, '> 2000 mg/kg (tikus)', '150 mg/kg/hari', 'C', 2, 1,
 'Hipoglikemia; hipotensi; kehamilan (uji terbatas)',
 'Penurunan gula darah berlebihan; diare; mual',
 '3 gram ekstrak standar/hari',
 '500-900 mg ekstrak/hari (terbagi 3 dosis)',
 'Hepatotoksisitas tidak signifikan pada dosis terapi',
 'Potensi hipoglikemik, monitor gula darah'),

(10, '> 2000 mg/kg (tikus)', '200 mg/kg/hari', 'B', 1, 1,
 'Hipersensitivitas Myrtaceae; kehamilan trimester 3 (data terbatas)',
 'Konstipasi (pada konsumsi buah berlebihan); alergi',
 '50 g daun rebus/hari',
 '15-30 g daun segar direbus dalam 500 ml air; 250-500 mg ekstrak/hari',
 'Tidak ada toksisitas organ signifikan pada dosis terapi',
 'Daun lebih digunakan medis, buah untuk konsumsi umum');

-- ============================================================
-- SEED DATA: REGULASI
-- ============================================================
INSERT INTO regulasi (tanaman_id, lembaga, status, nomor_registrasi, kategori_produk, klaim_disetujui, klaim_dilarang, batas_dosis_regulasi, tahun_regulasi, dokumen_referensi, catatan) VALUES
(1, 'BPOM', 'Approved', 'BPOM-OHT-001', 'OHT',
 'Mendukung kesehatan sendi; antioksidan; membantu menjaga fungsi hati normal',
 'Mengobati kanker; menyembuhkan penyakit liver; menggantikan obat resep',
 'Maks 500 mg kurkumin terstandar/hari untuk OHT', 2019,
 'PerBPOM No. 32 Tahun 2019', 'Produk berlogo OHT (daun dalam lingkaran) memenuhi standar uji praklinik dan klinik'),

(1, 'WHO', 'Traditional Medicine', NULL, 'Herbal Medicine Monograph',
 'Anti-inflammatory; antioxidant; digestive aid',
 'Disease treatment claims without clinical evidence',
 'Up to 3g dried rhizome daily', 2007,
 'WHO Monographs on Selected Medicinal Plants Vol. 1', 'Termasuk dalam daftar WHO Traditional Medicine Strategy'),

(2, 'BPOM', 'Approved', 'BPOM-OHT-002', 'OHT',
 'Meredakan mual dan gangguan pencernaan; antiemetik pada kehamilan muda',
 'Mengobati penyakit sistemik; klaim anti-kanker',
 'Maks 1 g/hari untuk kehamilan; maks 4 g/hari umum', 2019,
 'PerBPOM No. 32 Tahun 2019', NULL),

(3, 'BPOM', 'Approved', 'BPOM-OHT-003', 'OHT',
 'Membantu meredakan gejala infeksi saluran napas atas; imunomodulator',
 'DILARANG: Klaim pengobatan COVID-19 tanpa data; klaim untuk kehamilan',
 'Maks 200 mg andrografolid/hari; TIDAK untuk ibu hamil', 2020,
 'PerBPOM No. 32 Tahun 2019; SE BPOM 2020', 'Ditingkatkan statusnya saat pandemi COVID-19 dengan catatan'),

(7, 'BPOM', 'Approved', 'BPOM-OHT-007', 'OHT',
 'Imunomodulator; antioksidan; mendukung kesehatan umum',
 'Klaim mengobati kanker; menggantikan terapi medis konvensional',
 'Maks 3 gram biji bubuk/hari; maks 2 ml minyak/hari', 2018,
 'PerBPOM No. 32 Tahun 2019', NULL),

(8, 'BPOM', 'Approved', 'BPOM-OHT-008', 'OHT',
 'Diuretik; membantu kesehatan saluran kemih; mendukung fungsi ginjal',
 'Mengobati gagal ginjal; klaim setara diuretik resep',
 'Maks 3 gram daun kering/hari; tidak lebih dari 4 minggu tanpa pengawasan', 2019,
 'PerBPOM No. 32 Tahun 2019', 'Monitoring elektrolit diperlukan untuk penggunaan jangka panjang'),

(5, 'BPOM', 'Approved', 'BPOM-OHT-005', 'OHT',
 'Membantu penyembuhan luka (topikal); mendukung kesehatan kognitif',
 'Klaim mengobati Alzheimer; menggantikan obat resep neurologis',
 'Topikal: sesuai formulasi; Oral: maks 60 mg ekstrak standar/hari', 2019,
 'PerBPOM No. 32 Tahun 2019', NULL);

-- ============================================================
-- SEED DATA: CARA PENGOLAHAN
-- ============================================================
INSERT INTO cara_pengolahan (tanaman_id, metode, bagian_digunakan, takaran_bahan, volume_pelarut, suhu_proses, durasi_proses, cara_konsumsi, frekuensi, catatan_keamanan, level_pengguna) VALUES
(1, 'Rebusan tradisional', 'Rimpang segar', '3-5 cm rimpang', '400 ml air', 'Mendidih (100°C)', '10-15 menit', '100-200 ml saat hangat, setelah makan', '2x/hari', 'Kurangi dosis jika ada gangguan lambung', 'umum'),
(1, 'Minuman jamu', 'Rimpang segar', '1 rimpang (± 20g)', '250 ml air hangat', '60-80°C', 'Rendam 10 menit', '1 gelas pagi hari', '1x/hari', 'Konsumsi bersama madu untuk rasa', 'umum'),
(2, 'Rebusan jahe', 'Rimpang segar/kering', '2-3 cm atau 1 sdt bubuk', '300 ml air', 'Mendidih', '10 menit', '1 cangkir hangat setelah makan', '3x/hari', 'Batasi pada penderita GERD; hentikan jika ada heartburn', 'umum'),
(3, 'Kapsul standar', 'Daun kering (serbuk)', '200-400 mg ekstrak terstandar per kapsul', 'N/A', 'N/A', 'N/A', 'Dengan segelas air', '3x/hari (max 5 hari)', 'JANGAN untuk ibu hamil; hentikan jika tekanan darah turun berlebihan', 'medis'),
(5, 'Rebusan pegagan', 'Daun segar', 'Segenggam daun (± 15g)', '300 ml air', 'Mendidih', '10 menit', '1 gelas setelah makan', '2x/hari', 'Monitor fungsi hati jika digunakan > 4 minggu', 'umum'),
(7, 'Konsumsi biji langsung', 'Biji', '1 sendok teh (± 3g)', 'N/A', 'N/A', 'N/A', 'Dicampur madu atau yogurt', '1-2x/hari', 'Mulai dengan dosis kecil; hentikan jika ada reaksi alergi', 'umum'),
(8, 'Infus daun (teh herbal)', 'Daun kering', '1-2 gram daun kering', '200 ml air panas', '90-100°C', 'Seduh 5-10 menit', '1 cangkir sebelum makan', '2x/hari', 'Minum banyak air putih; monitor frekuensi BAK', 'umum'),
(10, 'Rebusan daun jambu', 'Daun muda segar', '5-8 lembar daun', '400 ml air', 'Mendidih', '15 menit', '100-200 ml, saat dingin', '3x/hari', 'Bisa menyebabkan konstipasi jika berlebihan', 'umum');

-- ============================================================
-- END OF SEED DATA
-- ============================================================

"""

with open("herbal_db.sql", "w", encoding="utf-8") as f:
    f.write(sql_content)
print("✅ herbal_db.sql berhasil ditulis!")
print(f"   Ukuran: {len(sql_content):,} karakter")


✅ herbal_db.sql berhasil ditulis!
   Ukuran: 36,971 karakter


In [3]:
import os, re, json, uuid, sqlite3, hashlib, time, textwrap, datetime
from pathlib import Path
from typing import Optional, Dict, List, Tuple, Any

import torch
import numpy as np
from dataclasses import dataclass, field
from enum import Enum

try:
    from rich.console import Console
    from rich.panel import Panel
    from rich.table import Table
    from rich.text import Text
    from rich.markdown import Markdown
    from rich import box
    RICH_AVAILABLE = True
except ImportError:
    RICH_AVAILABLE = False

console = Console() if RICH_AVAILABLE else None


class UserRole(Enum):
    PELAJAR      = "pelajar"
    UMUM         = "umum"
    MEDIS        = "medis"
    PENELITI     = "peneliti"
    UNKNOWN      = "unknown"

class ResponseType(Enum):
    ANSWERED     = "answered"
    REJECTED     = "rejected"
    PARTIAL      = "partial"
    CLARIFICATION = "clarification"

class GatekeeperFlag(Enum):
    CLEAR        = "CLEAR"
    HALLUCINATION_RISK = "HALLUCINATION_RISK"
    NO_DATA      = "NO_DATA"
    POLICY_VIOLATION = "POLICY_VIOLATION"
    MEDICAL_ADVICE = "MEDICAL_ADVICE"

@dataclass
class AgentConfig:
    """Konfigurasi global HERBALPHARMA-AGI"""
    model_name:        str   = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"
    max_seq_length:    int   = 2048
    max_new_tokens:    int   = 512
    db_path:           str   = "herbal_db.sqlite"
    sql_schema_path:   str   = "herbal_db.sql"
    embedding_model:   str   = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    device:            str   = "cuda" if torch.cuda.is_available() else "cpu"
    top_k_retrieval:   int   = 5
    audit_enabled:     bool  = True
    version:           str   = "1.0.0"

CONFIG = AgentConfig()

@dataclass
class SessionState:
    """Status sesi per user"""
    session_id:    str       = field(default_factory=lambda: str(uuid.uuid4())[:8])
    user_role:     UserRole  = UserRole.UNKNOWN
    role_locked:   bool      = False
    turn_count:    int       = 0
    history:       List[Dict] = field(default_factory=list)
    feedback_log:  List[Dict] = field(default_factory=list)

@dataclass
class QueryResult:
    """Hasil dari agen"""
    query:          str
    intent:         str
    sql_executed:   List[str]
    rag_context:    str
    raw_data:       List[Dict]
    llm_response:   str
    gatekeeper_flag: GatekeeperFlag
    response_type:  ResponseType
    sources:        List[str]
    confidence:     float
    processing_time: float


class HerbalDatabase:
    """
    Manajemen database SQLite — Single Source of Truth.
    Semua jawaban HARUS bersumber dari sini.
    """

    STANDARD_REJECTION = (
        "⚠️ DATA TIDAK TERSEDIA DALAM DATABASE.\n\n"
        "Sistem hanya menjawab berdasarkan data terverifikasi dalam basis data HERBALPHARMA-AGI. "
        "Informasi yang Anda tanyakan tidak ditemukan. "
        "Untuk informasi lebih lanjut, konsultasikan dengan tenaga ahli farmasi atau dokter."
    )

    def __init__(self, db_path: str, sql_schema: str):
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path, check_same_thread=False)
        self.conn.row_factory = sqlite3.Row
        self._init_database(sql_schema)

    def _init_database(self, schema_path: str):
        """Inisialisasi database dari file SQL schema"""
        if Path(schema_path).exists():
            with open(schema_path, 'r', encoding='utf-8') as f:
                sql = f.read()
            cursor = self.conn.cursor()
            # Eksekusi per statement
            statements = [s.strip() for s in sql.split(';') if s.strip()
                         and not s.strip().startswith('--')]
            for stmt in statements:
                try:
                    cursor.execute(stmt)
                except sqlite3.Error:
                    pass  # Skip jika sudah ada
            self.conn.commit()
        else:
            print(f"[WARNING] Schema {schema_path} tidak ditemukan. Gunakan database yang sudah ada.")

    def execute_query(self, query: str, params: tuple = ()) -> Tuple[List[Dict], bool]:
        """Eksekusi query SQL yang aman (read-only)"""
        # Keamanan: hanya izinkan SELECT
        clean_q = query.strip().upper()
        if not clean_q.startswith('SELECT'):
            return [], False
        try:
            cursor = self.conn.cursor()
            cursor.execute(query, params)
            rows = [dict(row) for row in cursor.fetchall()]
            return rows, True
        except sqlite3.Error as e:
            print(f"[DB ERROR] {e}")
            return [], False

    def write_query(self, query: str, params: tuple = ()) -> bool:
        """Eksekusi query tulis (INSERT only, untuk audit & feedback)"""
        clean_q = query.strip().upper()
        if not any(clean_q.startswith(k) for k in ['INSERT', 'UPDATE']):
            return False
        try:
            cursor = self.conn.cursor()
            cursor.execute(query, params)
            self.conn.commit()
            return True
        except sqlite3.Error as e:
            print(f"[DB WRITE ERROR] {e}")
            return False


    def search_plant_by_name(self, name: str) -> List[Dict]:
        """Cari tanaman berdasarkan nama (ilmiah, lokal, inggris)"""
        q = """
        SELECT t.*,
               GROUP_CONCAT(DISTINCT s.nama_senyawa) as senyawa_list,
               GROUP_CONCAT(DISTINCT ef.nama_efek) as efek_list
        FROM tanaman t
        LEFT JOIN tanaman_senyawa ts ON t.id = ts.tanaman_id
        LEFT JOIN senyawa s ON ts.senyawa_id = s.id
        LEFT JOIN tanaman_efek te ON t.id = te.tanaman_id
        LEFT JOIN efek_farmakologi ef ON te.efek_id = ef.id
        WHERE LOWER(t.nama_ilmiah) LIKE LOWER(?)
           OR LOWER(t.nama_lokal) LIKE LOWER(?)
           OR LOWER(t.nama_inggris) LIKE LOWER(?)
        GROUP BY t.id
        """
        pattern = f"%{name}%"
        rows, _ = self.execute_query(q, (pattern, pattern, pattern))
        return rows

    def get_plant_effects(self, tanaman_id: int, level_filter: str = None) -> List[Dict]:
        """Ambil efek farmakologi dengan level bukti"""
        q = """
        SELECT ef.nama_efek, ef.kategori, ef.mekanisme_umum,
               te.level_bukti, te.dosis_efektif, te.rute_pemberian,
               te.populasi_studi, te.referensi, te.catatan
        FROM tanaman_efek te
        JOIN efek_farmakologi ef ON te.efek_id = ef.id
        WHERE te.tanaman_id = ?
        """
        params = [tanaman_id]
        if level_filter:
            q += " AND te.level_bukti = ?"
            params.append(level_filter)
        q += " ORDER BY te.level_bukti ASC"
        rows, _ = self.execute_query(q, tuple(params))
        return rows

    def get_plant_compounds(self, tanaman_id: int) -> List[Dict]:
        """Ambil senyawa kimia tanaman"""
        q = """
        SELECT s.nama_senyawa, s.golongan, s.rumus_molekul, s.berat_molekul,
               s.mekanisme_aksi, s.bioavailabilitas,
               ts.kadar_persentase, ts.bagian_tanaman, ts.metode_ekstraksi, ts.referensi
        FROM tanaman_senyawa ts
        JOIN senyawa s ON ts.senyawa_id = s.id
        WHERE ts.tanaman_id = ?
        ORDER BY ts.kadar_persentase DESC
        """
        rows, _ = self.execute_query(q, (tanaman_id,))
        return rows

    def get_safety_info(self, tanaman_id: int) -> List[Dict]:
        """Ambil data keamanan & toksikologi"""
        q = """
        SELECT * FROM keamanan_toksikologi WHERE tanaman_id = ?
        """
        rows, _ = self.execute_query(q, (tanaman_id,))
        return rows

    def get_drug_interactions(self, tanaman_id: int) -> List[Dict]:
        """Ambil interaksi obat"""
        q = """
        SELECT * FROM interaksi_obat WHERE tanaman_id = ?
        ORDER BY CASE tingkat_keparahan
            WHEN 'Contraindicated' THEN 1
            WHEN 'Major' THEN 2
            WHEN 'Moderate' THEN 3
            WHEN 'Minor' THEN 4 ELSE 5 END
        """
        rows, _ = self.execute_query(q, (tanaman_id,))
        return rows

    def get_clinical_studies(self, tanaman_id: int, role: UserRole = UserRole.UMUM) -> List[Dict]:
        """Ambil studi klinis, filter berdasarkan role"""
        q = """
        SELECT * FROM studi_klinis WHERE tanaman_id = ?
        ORDER BY level_evidens ASC, tahun DESC
        """
        rows, _ = self.execute_query(q, (tanaman_id,))
        return rows

    def get_regulatory_info(self, tanaman_id: int) -> List[Dict]:
        """Ambil info regulasi BPOM/WHO/EMA"""
        q = """
        SELECT * FROM regulasi WHERE tanaman_id = ?
        ORDER BY lembaga
        """
        rows, _ = self.execute_query(q, (tanaman_id,))
        return rows

    def get_processing_methods(self, tanaman_id: int, level_pengguna: str = "umum") -> List[Dict]:
        """Ambil cara pengolahan sesuai level pengguna"""
        q = """
        SELECT * FROM cara_pengolahan
        WHERE tanaman_id = ? AND (level_pengguna = ? OR level_pengguna = 'umum')
        """
        rows, _ = self.execute_query(q, (tanaman_id, level_pengguna))
        return rows

    def get_metabolic_pathways(self, tanaman_id: int) -> List[Dict]:
        """Ambil jalur metabolit untuk peneliti/medis"""
        q = """
        SELECT s.nama_senyawa, s.golongan, jm.*
        FROM jalur_metabolit jm
        JOIN senyawa s ON jm.senyawa_id = s.id
        JOIN tanaman_senyawa ts ON s.id = ts.senyawa_id
        WHERE ts.tanaman_id = ?
        """
        rows, _ = self.execute_query(q, (tanaman_id,))
        return rows

    def search_by_effect(self, efek: str) -> List[Dict]:
        """Cari tanaman berdasarkan efek farmakologi"""
        q = """
        SELECT t.nama_ilmiah, t.nama_lokal, t.nama_inggris,
               ef.nama_efek, ef.kategori,
               te.level_bukti, te.dosis_efektif, te.referensi
        FROM tanaman_efek te
        JOIN tanaman t ON te.tanaman_id = t.id
        JOIN efek_farmakologi ef ON te.efek_id = ef.id
        WHERE LOWER(ef.nama_efek) LIKE LOWER(?)
           OR LOWER(ef.kategori) LIKE LOWER(?)
        ORDER BY te.level_bukti ASC
        """
        pattern = f"%{efek}%"
        rows, _ = self.execute_query(q, (pattern, pattern))
        return rows

    def log_audit(self, session_id: str, user_role: str, query: str,
                  intent: str, sql: str, resp_type: str, flag: str):
        """Tulis ke audit log"""
        q = """
        INSERT INTO audit_log (session_id, user_role, query_text, intent_detected,
                               sql_executed, response_type, gatekeeper_flag, timestamp)
        VALUES (?, ?, ?, ?, ?, ?, ?, datetime('now'))
        """
        self.write_query(q, (session_id, user_role, query, intent, sql, resp_type, flag))

    def log_feedback(self, session_id: str, query: str, response: str,
                     rating: int, feedback: str, role: str):
        """Simpan feedback user untuk pembelajaran"""
        q = """
        INSERT INTO feedback_log (session_id, query_text, response_text,
                                  rating, feedback_text, user_role, timestamp)
        VALUES (?, ?, ?, ?, ?, ?, datetime('now'))
        """
        self.write_query(q, (session_id, query, response, rating, feedback, role))

    def get_feedback_summary(self) -> Dict:
        """Statistik feedback untuk monitoring"""
        q = "SELECT user_role, AVG(rating) as avg_rating, COUNT(*) as total FROM feedback_log GROUP BY user_role"
        rows, _ = self.execute_query(q)
        return {r['user_role']: {'avg_rating': r['avg_rating'], 'total': r['total']} for r in rows}



In [4]:
class RoleDetector:
    """
    Deteksi role user secara otomatis berdasarkan konteks pertanyaan.
    Mendukung mode eksplisit (user menyatakan role) dan implisit (dari pola bahasa).
    """

    # Pattern eksplisit: user menyebutkan role secara langsung
    EXPLICIT_PATTERNS = {
        UserRole.PELAJAR: [
            r'\bpelajar\b', r'\bsiswa\b', r'\bmahasiswa\b', r'\btugas\b',
            r'\bsekolah\b', r'\bsma\b', r'\bsmp\b', r'\bbelajar\b',
            r'saya (adalah |seorang )?pelajar', r'untuk (tugas|pembelajaran)',
        ],
        UserRole.UMUM: [
            r'\bumum\b', r'masyarakat umum', r'orang awam',
            r'saya bukan (dokter|apoteker|tenaga medis)',
            r'untuk (keluarga|pribadi|rumahan)',
            r'cara (minum|makan|memasak|membuat)',
        ],
        UserRole.MEDIS: [
            r'\bdokter\b', r'\bapoteker\b', r'\bperawat\b', r'\bbidan\b',
            r'\bfarmasi\b', r'\btenaga (medis|kesehatan)\b',
            r'\bklinisi\b', r'\bpraktisi\b',
            r'(pasien|dosis klinis|terapi|drug interaction)',
            r'saya (adalah |bekerja sebagai )?(dokter|apoteker|tenaga)',
        ],
        UserRole.PENELITI: [
            r'\bpeneliti\b', r'\briset\b', r'\bpenelitian\b', r'\beksperimen\b',
            r'\bprofessor\b', r'\bdosen\b', r'\blab\b', r'\blaboratorium\b',
            r'\bin vitro\b', r'\bin vivo\b', r'\buji klinis\b',
            r'(fitokimia|farmakognosi|metabolomik|genomik)',
            r'saya (adalah |seorang )?(peneliti|akademisi|ilmuwan)',
        ],
    }

    # Pattern implisit: berdasarkan gaya pertanyaan
    IMPLICIT_SIGNALS = {
        UserRole.PELAJAR: [
            r'apa itu', r'jelaskan', r'pengertian', r'fungsi (dari )?',
            r'kenapa\b', r'mengapa\b', r'bagaimana cara kerja',
        ],
        UserRole.UMUM: [
            r'cara (membuat|memasak|merebus|minum)',
            r'manfaat (untuk|bagi)', r'aman (tidak|kah)',
            r'boleh (tidak|kah)', r'berapa (banyak|kali)',
            r'apakah (berbahaya|aman|boleh)',
        ],
        UserRole.MEDIS: [
            r'dosis (terapi|klinis|maksimal)', r'kontraindikasi',
            r'interaksi (obat|farmaka)', r'mekanisme aksi',
            r'farmakokinetik', r'farmakodinamik',
            r'efek samping klinis', r'evidence.based',
        ],
        UserRole.PENELITI: [
            r'struktur kimia', r'isolasi senyawa', r'ekstraksi (metode|protokol)',
            r'jalur metabol', r'signaling pathway', r'ic50\b', r'ec50\b',
            r'uji (fitokimia|toksisitas|ames)', r'kromatografi',
            r'spektroskopi', r'nmr\b', r'gcms\b', r'hplc\b',
            r'(bioavailabilitas|bioaktivitas) (senyawa|komponen)',
        ],
    }

    def detect_role(self, text: str, current_role: UserRole = UserRole.UNKNOWN) -> Tuple[UserRole, float]:
        """
        Deteksi role dari teks dengan confidence score.
        Returns: (UserRole, confidence_score)
        """
        text_lower = text.lower()
        scores = {role: 0.0 for role in UserRole}

        # Cek pattern eksplisit (bobot tinggi)
        for role, patterns in self.EXPLICIT_PATTERNS.items():
            for pattern in patterns:
                if re.search(pattern, text_lower):
                    scores[role] += 3.0

        # Cek pattern implisit (bobot lebih rendah)
        for role, patterns in self.IMPLICIT_SIGNALS.items():
            for pattern in patterns:
                if re.search(pattern, text_lower):
                    scores[role] += 1.0

        # Exclude UNKNOWN dari pemilihan
        relevant = {k: v for k, v in scores.items() if k != UserRole.UNKNOWN and v > 0}

        if not relevant:
            return current_role if current_role != UserRole.UNKNOWN else UserRole.UMUM, 0.3

        best_role = max(relevant, key=relevant.get)
        total = sum(relevant.values())
        confidence = relevant[best_role] / total if total > 0 else 0.5

        return best_role, confidence

    def format_role_greeting(self, role: UserRole) -> str:
        greetings = {
            UserRole.PELAJAR: (
                "🎓 Mode **PELAJAR** aktif.\n"
                "Saya akan menjelaskan dengan bahasa yang mudah dipahami untuk pembelajaran tingkat SMA. "
                "Jawaban saya akan fokus pada pemahaman dasar ilmu farmakognosi dan botani."
            ),
            UserRole.UMUM: (
                "👤 Mode **UMUM** aktif.\n"
                "Saya siap memberikan informasi tentang manfaat, cara pengolahan, dan keamanan tanaman herbal "
                "dalam bahasa yang praktis dan mudah dipahami."
            ),
            UserRole.MEDIS: (
                "🩺 Mode **TENAGA MEDIS/FARMASI** aktif.\n"
                "Saya akan menyajikan data klinis lengkap termasuk mekanisme aksi, farmakokinetik, "
                "interaksi obat, dan referensi ilmiah untuk mendukung praktik klinis Anda."
            ),
            UserRole.PENELITI: (
                "🔬 Mode **PENELITI/INDUSTRI** aktif.\n"
                "Akses penuh ke data fitokimia, jalur metabolit, studi klinis, data toksikologi, "
                "informasi regulasi, dan referensi ilmiah setara level professor untuk mendukung riset Anda."
            ),
        }
        return greetings.get(role, "Mode umum aktif.")


class IntentDetector:
    """
    Mendeteksi intent dari query user dan merencanakan query SQL yang relevan.
    """

    INTENT_PATTERNS = {
        "cari_tanaman": [
            r'apa (itu|manfaat) (.+)',
            r'(tentang|informasi|info) (.+)',
            r'(kunyit|jahe|sambiloto|mengkudu|pegagan|kencur|jintan|kumis kucing|meniran|jambu)',
        ],
        "efek_farmakologi": [
            r'(manfaat|khasiat|kegunaan|efek|aktivitas)',
            r'(antiinflamasi|antioksidan|antimikroba|antidiabetes|hepatoprotektif)',
            r'(bisa|dapat) (mengobati|membantu|meredakan)',
        ],
        "keamanan_dosis": [
            r'(aman|bahaya|efek samping|kontraindikasi)',
            r'(boleh|tidak boleh|dilarang|hindari)',
            r'(dosis|takaran|berapa)',
            r'(hamil|menyusui|anak|lansia)',
        ],
        "interaksi_obat": [
            r'(interaksi|bersamaan|kombinasi)',
            r'(obat|warfarin|aspirin|metformin|antihipertensi)',
            r'boleh diminum (bersama|dengan)',
        ],
        "cara_pengolahan": [
            r'(cara|metode|bagaimana) (membuat|mengolah|merebus|minum|menggunakan)',
            r'(resep|takaran|bahan)',
            r'(direbus|diseduh|dikapsul|ekstrak)',
        ],
        "senyawa_kimia": [
            r'(senyawa|komponen|fitokimia|kandungan kimia)',
            r'(kurkumin|gingerol|andrografolid|flavonoid|alkaloid)',
            r'(struktur|rumus|molecular)',
        ],
        "studi_klinis": [
            r'(studi|penelitian|bukti|evidens|clinical trial|rct)',
            r'(terbukti|evidence.based|efektivitas klinis)',
            r'(jurnal|publikasi|doi)',
        ],
        "regulasi": [
            r'(bpom|who|ema|fda|regulasi|registrasi)',
            r'(terdaftar|legal|izin edar|sertifikasi)',
            r'(jamu|oht|fitofarmaka|suplemen)',
            r'(klaim|label)',
        ],
        "jalur_metabolit": [
            r'(farmakokinetik|farmakodinamik|metabolisme|metabolit)',
            r'(cyp|p450|bioavailabilitas|half.life|t1/2)',
            r'(absorpsi|distribusi|eliminasi|ekskresi)',
        ],
        "cari_berdasarkan_efek": [
            r'tanaman (yang|untuk|bisa|dapat)',
            r'(herbal|obat) (untuk|yang bisa|anti)',
            r'(rekomendasi|rekomendasikan)',
        ],
    }

    def detect_intent(self, query: str) -> Tuple[str, float]:
        """Deteksi intent utama dari query"""
        query_lower = query.lower()
        scores = {intent: 0 for intent in self.INTENT_PATTERNS}

        for intent, patterns in self.INTENT_PATTERNS.items():
            for pattern in patterns:
                if re.search(pattern, query_lower):
                    scores[intent] += 1

        if not any(scores.values()):
            return "umum", 0.3

        best = max(scores, key=scores.get)
        total = sum(scores.values())
        conf = scores[best] / total if total > 0 else 0.5
        return best, conf

    def extract_plant_name(self, query: str) -> Optional[str]:
        """Ekstrak nama tanaman dari query"""
        known_plants = {
            'kunyit': 'kunyit', 'turmeric': 'kunyit', 'curcuma': 'kunyit',
            'jahe': 'jahe', 'ginger': 'jahe', 'zingiber': 'jahe',
            'sambiloto': 'sambiloto', 'andrographis': 'sambiloto',
            'mengkudu': 'mengkudu', 'noni': 'mengkudu', 'morinda': 'mengkudu',
            'pegagan': 'pegagan', 'centella': 'pegagan', 'gotu kola': 'pegagan',
            'kencur': 'kencur', 'kaempferia': 'kencur',
            'jintan hitam': 'jintan hitam', 'black seed': 'jintan hitam', 'nigella': 'jintan hitam',
            'kumis kucing': 'kumis kucing', 'orthosiphon': 'kumis kucing',
            'meniran': 'meniran', 'phyllanthus': 'meniran',
            'jambu biji': 'jambu biji', 'jambu': 'jambu biji', 'guava': 'jambu biji',
        }
        query_lower = query.lower()
        for keyword, plant in known_plants.items():
            if keyword in query_lower:
                return plant
        return None

    def extract_effect_name(self, query: str) -> Optional[str]:
        """Ekstrak nama efek farmakologi dari query"""
        effects = [
            'antiinflamasi', 'antioksidan', 'antimikroba', 'antidiabetes',
            'hepatoprotektif', 'imunomodulator', 'antikanker', 'antihipertensi',
            'diuretik', 'neuroprotektif', 'antiemetik', 'penyembuhan luka',
            'anti inflamasi', 'anti oksidan', 'anti kanker', 'anti diabetes',
        ]
        query_lower = query.lower()
        for effect in effects:
            if effect in query_lower:
                return effect
        return None



In [5]:
class GNavAgent:
    """
    Agen Navigator Graf Klinis (G-Nav).
    Menavigasi relasi antar entitas dalam knowledge graph:
    tanaman → senyawa → efek → jalur metabolit → studi klinis
    """

    def __init__(self, db: HerbalDatabase):
        self.db = db
        self.intent_detector = IntentDetector()

    def navigate(self, query: str, role: UserRole) -> Dict[str, Any]:
        """
        Utama: navigasi graf berdasarkan query dan role.
        Returns dict berisi semua data relevan beserta SQL yang dijalankan.
        """
        intent, conf = self.intent_detector.detect_intent(query)
        plant_name = self.intent_detector.extract_plant_name(query)
        effect_name = self.intent_detector.extract_effect_name(query)

        result = {
            "intent": intent,
            "confidence": conf,
            "plant_name": plant_name,
            "effect_name": effect_name,
            "data": {},
            "sql_log": [],
            "sources": [],
            "data_found": False,
        }

        # ── Node 0: Resolve Tanaman ──────────────────────────────
        tanaman_data = []
        if plant_name:
            tanaman_data = self.db.search_plant_by_name(plant_name)
            result["sql_log"].append(f"search_plant_by_name('{plant_name}')")

        if tanaman_data:
            result["data_found"] = True
            tanaman = tanaman_data[0]
            tanaman_id = tanaman['id']
            result["data"]["tanaman"] = tanaman

            # ── Node 1: Senyawa ──────────────────────────────────
            if role in [UserRole.MEDIS, UserRole.PENELITI, UserRole.PELAJAR]:
                senyawa = self.db.get_plant_compounds(tanaman_id)
                result["data"]["senyawa"] = senyawa
                result["sql_log"].append(f"get_plant_compounds({tanaman_id})")
                if senyawa:
                    result["sources"].extend([s.get('referensi', '') for s in senyawa if s.get('referensi')])

            # ── Node 2: Efek Farmakologi ─────────────────────────
            efek = self.db.get_plant_effects(tanaman_id)
            result["data"]["efek"] = efek
            result["sql_log"].append(f"get_plant_effects({tanaman_id})")
            if efek:
                result["sources"].extend([e.get('referensi', '') for e in efek if e.get('referensi')])

            # ── Node 3: Keamanan & Toksikologi ──────────────────
            safety = self.db.get_safety_info(tanaman_id)
            result["data"]["safety"] = safety
            result["sql_log"].append(f"get_safety_info({tanaman_id})")

            # ── Node 4: Interaksi Obat (Medis & Peneliti) ────────
            if role in [UserRole.MEDIS, UserRole.PENELITI]:
                interactions = self.db.get_drug_interactions(tanaman_id)
                result["data"]["interaksi"] = interactions
                result["sql_log"].append(f"get_drug_interactions({tanaman_id})")
                if interactions:
                    result["sources"].extend([i.get('referensi', '') for i in interactions if i.get('referensi')])

            # ── Node 5: Studi Klinis ──────────────────────────────
            if role in [UserRole.MEDIS, UserRole.PENELITI]:
                studies = self.db.get_clinical_studies(tanaman_id, role)
                result["data"]["studi_klinis"] = studies
                result["sql_log"].append(f"get_clinical_studies({tanaman_id})")
                if studies:
                    result["sources"].extend([f"{s.get('jurnal', '')} ({s.get('tahun', '')})" for s in studies])

            # ── Node 6: Regulasi ──────────────────────────────────
            if intent in ["regulasi", "efek_farmakologi"] or role in [UserRole.MEDIS, UserRole.PENELITI]:
                regulasi = self.db.get_regulatory_info(tanaman_id)
                result["data"]["regulasi"] = regulasi
                result["sql_log"].append(f"get_regulatory_info({tanaman_id})")

            # ── Node 7: Cara Pengolahan ───────────────────────────
            if intent in ["cara_pengolahan", "keamanan_dosis"] or role in [UserRole.UMUM, UserRole.PELAJAR]:
                level = "medis" if role == UserRole.MEDIS else "umum"
                cara = self.db.get_processing_methods(tanaman_id, level)
                result["data"]["cara_pengolahan"] = cara
                result["sql_log"].append(f"get_processing_methods({tanaman_id}, '{level}')")

            # ── Node 8: Jalur Metabolit (Peneliti) ───────────────
            if role == UserRole.PENELITI or intent == "jalur_metabolit":
                jalur = self.db.get_metabolic_pathways(tanaman_id)
                result["data"]["jalur_metabolit"] = jalur
                result["sql_log"].append(f"get_metabolic_pathways({tanaman_id})")

        # ── Pencarian berdasarkan efek ───────────────────────────
        elif effect_name:
            efek_results = self.db.search_by_effect(effect_name)
            result["sql_log"].append(f"search_by_effect('{effect_name}')")
            if efek_results:
                result["data_found"] = True
                result["data"]["tanaman_by_efek"] = efek_results

        return result


class DSynAgent:
    """
    Agen Orkestrator Data Terkontrol (D-Syn).
    Mengolah data dari G-Nav dan menyiapkan konteks terstruktur
    untuk input ke LLM Brain.
    """

    ROLE_INSTRUCTIONS = {
        UserRole.PELAJAR: """
INSTRUKSI UNTUK PELAJAR (SMA):
- Gunakan bahasa Indonesia yang sederhana dan mudah dipahami
- Jelaskan istilah ilmiah dengan analogi atau contoh sehari-hari
- Fokus pada: nama tanaman, kandungan utama, manfaat umum, dan cara penggunaan dasar
- JANGAN menyebut dosis klinis spesifik atau istilah farmakologi kompleks
- Buat penjelasan menarik dan edukatif
- Panjang jawaban: 200-350 kata
""",
        UserRole.UMUM: """
INSTRUKSI UNTUK MASYARAKAT UMUM:
- Gunakan bahasa Indonesia sehari-hari yang praktis
- Fokus pada: manfaat praktis, cara pengolahan/penggunaan, dan catatan keamanan penting
- Sertakan peringatan keamanan yang jelas dan mudah dipahami
- JANGAN berikan saran medis spesifik atau menggantikan rekomendasi dokter
- Sarankan konsultasi dokter/apoteker untuk kondisi medis tertentu
- Panjang jawaban: 250-400 kata
""",
        UserRole.MEDIS: """
INSTRUKSI UNTUK TENAGA MEDIS/FARMASI:
- Gunakan terminologi klinis yang tepat
- Sajikan: mekanisme aksi, farmakokinetik, dosis terapi, interaksi obat, kontraindikasi klinis
- Sertakan level evidens dari studi klinis yang tersedia
- Berikan perspektif klinis yang praktis untuk pengambilan keputusan terapeutik
- Kutip referensi ilmiah yang relevan
- Sebutkan status regulasi BPOM/WHO yang berlaku
- Panjang jawaban: 350-600 kata
""",
        UserRole.PENELITI: """
INSTRUKSI UNTUK PENELITI/INDUSTRI:
- Sajikan data ilmiah lengkap dan komprehensif
- Inklusikan: struktur kimia, jalur biosintesis, jalur metabolit (CYP enzyme), data farmakokinetik
- Tampilkan semua studi klinis tersedia beserta metodologi dan limitasi
- Bahas potensi pengembangan fitofarmaka dan area riset yang masih terbuka
- Sertakan informasi regulasi lengkap (BPOM, WHO, EMA) untuk pengembangan produk
- Kutip referensi ilmiah dengan jurnal dan DOI
- Analisis gap pengetahuan dan saran riset lanjutan
- Panjang jawaban: 500-800 kata
""",
    }

    def synthesize_context(self, nav_result: Dict, role: UserRole, query: str) -> str:
        """
        Buat konteks terstruktur untuk LLM dari hasil G-Nav.
        """
        if not nav_result["data_found"]:
            return ""

        ctx_parts = []
        data = nav_result["data"]
        role_instr = self.ROLE_INSTRUCTIONS.get(role, self.ROLE_INSTRUCTIONS[UserRole.UMUM])

        ctx_parts.append(role_instr)
        ctx_parts.append(f"\n=== PERTANYAAN USER ===\n{query}\n")
        ctx_parts.append("=== DATA TERVERIFIKASI DARI DATABASE ===\n")

        # ── Informasi Tanaman Utama ────────────────────────────
        if "tanaman" in data:
            t = data["tanaman"]
            ctx_parts.append(f"""
[INFORMASI TANAMAN]
Nama Ilmiah    : {t.get('nama_ilmiah', 'N/A')}
Nama Lokal     : {t.get('nama_lokal', 'N/A')}
Nama Inggris   : {t.get('nama_inggris', 'N/A')}
Familia        : {t.get('familia', 'N/A')}
Bagian Digunakan: {t.get('bagian_digunakan', 'N/A')}
Status BPOM    : {t.get('status_bpom', 'N/A')}
Status WHO     : {t.get('status_who', 'N/A')}
Deskripsi      : {t.get('deskripsi', 'N/A')}
Catatan Keamanan: {t.get('catatan_keamanan', 'N/A')}
""")

        # ── Senyawa Kimia ──────────────────────────────────────
        if "senyawa" in data and data["senyawa"]:
            ctx_parts.append("[KANDUNGAN SENYAWA AKTIF]")
            for s in data["senyawa"]:
                ctx_parts.append(
                    f"• {s.get('nama_senyawa')} ({s.get('golongan')}) | "
                    f"Kadar: {s.get('kadar_persentase', '?')}% | "
                    f"Bagian: {s.get('bagian_tanaman')} | "
                    f"Mekanisme: {s.get('mekanisme_aksi', 'N/A')} | "
                    f"Bioavailabilitas: {s.get('bioavailabilitas', 'N/A')} | "
                    f"Ref: {s.get('referensi', 'N/A')}"
                )
            ctx_parts.append("")

        # ── Efek Farmakologi ───────────────────────────────────
        if "efek" in data and data["efek"]:
            ctx_parts.append("[EFEK FARMAKOLOGI TERVERIFIKASI]")
            for e in data["efek"]:
                ctx_parts.append(
                    f"• {e.get('nama_efek')} | Level Bukti: {e.get('level_bukti')} | "
                    f"Dosis Efektif: {e.get('dosis_efektif', 'N/A')} | "
                    f"Rute: {e.get('rute_pemberian', 'N/A')} | "
                    f"Populasi: {e.get('populasi_studi', 'N/A')} | "
                    f"Ref: {e.get('referensi', 'N/A')}"
                )
            ctx_parts.append("")

        # ── Keamanan ───────────────────────────────────────────
        if "safety" in data and data["safety"]:
            s = data["safety"][0]
            ctx_parts.append(f"""[KEAMANAN & TOKSIKOLOGI]
LD50               : {s.get('ld50', 'N/A')}
Kategori Kehamilan : {s.get('kategori_kehamilan', 'N/A')} | Aman Laktasi: {'Ya' if s.get('aman_laktasi') == 1 else 'Data terbatas' if s.get('aman_laktasi') == 2 else 'TIDAK'}
Kontraindikasi     : {s.get('kontraindikasi', 'N/A')}
Efek Samping       : {s.get('efek_samping', 'N/A')}
Dosis Terapi Umum  : {s.get('dosis_terapi_umum', 'N/A')}
Dosis Maksimal     : {s.get('dosis_maksimal_harian', 'N/A')}
Catatan Khusus     : {s.get('catatan_khusus', 'N/A')}
""")

        # ── Interaksi Obat ─────────────────────────────────────
        if "interaksi" in data and data["interaksi"]:
            ctx_parts.append("[INTERAKSI OBAT - PENTING]")
            for i in data["interaksi"]:
                severity_emoji = "🔴" if i.get('tingkat_keparahan') in ['Major', 'Contraindicated'] else "🟡"
                ctx_parts.append(
                    f"{severity_emoji} {i.get('obat_konvensional')} | "
                    f"Tingkat: {i.get('tingkat_keparahan')} | "
                    f"Mekanisme: {i.get('mekanisme', 'N/A')} | "
                    f"Efek: {i.get('efek_interaksi', 'N/A')} | "
                    f"Rekomendasi: {i.get('rekomendasi_klinis', 'N/A')}"
                )
            ctx_parts.append("")

        # ── Studi Klinis ───────────────────────────────────────
        if "studi_klinis" in data and data["studi_klinis"]:
            ctx_parts.append("[STUDI KLINIS TERSEDIA]")
            for sc in data["studi_klinis"][:3]:  # Max 3 studi
                ctx_parts.append(
                    f"• [{sc.get('tipe_studi')}] {sc.get('judul_studi')} | "
                    f"n={sc.get('ukuran_sampel')} | Level {sc.get('level_evidens')} | "
                    f"{sc.get('jurnal')} {sc.get('tahun')} | "
                    f"Hasil: {sc.get('hasil', 'N/A')}"
                )
            ctx_parts.append("")

        # ── Regulasi ───────────────────────────────────────────
        if "regulasi" in data and data["regulasi"]:
            ctx_parts.append("[STATUS REGULASI]")
            for r in data["regulasi"]:
                ctx_parts.append(
                    f"• {r.get('lembaga')}: {r.get('status')} | Kategori: {r.get('kategori_produk', 'N/A')} | "
                    f"Klaim Disetujui: {r.get('klaim_disetujui', 'N/A')} | "
                    f"Klaim DILARANG: {r.get('klaim_dilarang', 'N/A')}"
                )
            ctx_parts.append("")

        # ── Cara Pengolahan ────────────────────────────────────
        if "cara_pengolahan" in data and data["cara_pengolahan"]:
            ctx_parts.append("[CARA PENGOLAHAN/PENGGUNAAN]")
            for cp in data["cara_pengolahan"]:
                ctx_parts.append(
                    f"• {cp.get('metode')}: {cp.get('takaran_bahan')} | "
                    f"Cara: {cp.get('cara_konsumsi')} | "
                    f"Frekuensi: {cp.get('frekuensi')} | "
                    f"Keamanan: {cp.get('catatan_keamanan', 'N/A')}"
                )
            ctx_parts.append("")

        # ── Jalur Metabolit (Peneliti) ─────────────────────────
        if "jalur_metabolit" in data and data["jalur_metabolit"]:
            ctx_parts.append("[JALUR METABOLIT - FARMAKOKINETIK]")
            for jm in data["jalur_metabolit"]:
                ctx_parts.append(
                    f"• {jm.get('nama_senyawa')} | CYP: {jm.get('enzim_cyp', 'N/A')} | "
                    f"Tipe: {jm.get('tipe_interaksi')} | T½: {jm.get('waktu_paruh', 'N/A')} | "
                    f"Protein Binding: {jm.get('protein_binding', 'N/A')}% | "
                    f"Metabolit: {jm.get('metabolit_utama', 'N/A')}"
                )
            ctx_parts.append("")

        # ── Tanaman berdasarkan efek ───────────────────────────
        if "tanaman_by_efek" in data and data["tanaman_by_efek"]:
            ctx_parts.append(f"[TANAMAN DENGAN EFEK TERKAIT]")
            for tb in data["tanaman_by_efek"][:6]:
                ctx_parts.append(
                    f"• {tb.get('nama_lokal')} ({tb.get('nama_ilmiah')}) | "
                    f"Efek: {tb.get('nama_efek')} | Level Bukti: {tb.get('level_bukti')} | "
                    f"Dosis: {tb.get('dosis_efektif', 'N/A')}"
                )
            ctx_parts.append("")

        # Sumber referensi
        sources = list(set([s for s in nav_result.get("sources", []) if s]))
        if sources:
            ctx_parts.append("[REFERENSI ILMIAH]")
            for src in sources[:5]:
                ctx_parts.append(f"• {src}")

        ctx_parts.append("\n=== INSTRUKSI AKHIR ===")
        ctx_parts.append(
            "Berikan jawaban berdasarkan HANYA data di atas. "
            "JANGAN tambahkan informasi di luar database. "
            "JANGAN berikan saran medis spesifik. "
            "Jika data tidak cukup untuk menjawab, nyatakan dengan jelas."
        )

        return "\n".join(ctx_parts)



In [6]:
class FinalGatekeeper:
    """
    Final Gatekeeper: Kepatuhan Semantik (Regulatory-First).
    Tiga pemeriksaan wajib:
    1. No-Hallucination Policy
    2. Kepatuhan Medis (Bukan Saran Langsung)
    3. Verifikasi Regulasi (BPOM)
    """

    # Kata-kata yang mengindikasikan klaim berlebihan / hallucination
    HALLUCINATION_TRIGGERS = [
        r'terbukti (100%|secara pasti|mutlak) (menyembuhkan|mengobati)',
        r'obat (mujarab|manjur|ajaib)',
        r'(menyembuhkan|membasmi) (kanker|tumor|HIV|AIDS|diabetes)',
        r'gantikan (obat|terapi) dokter',
        r'tidak perlu (konsultasi|ke dokter)',
        r'dijamin (aman|sembuh|berhasil)',
        r'(100%|sepenuhnya) (aman|terbukti)',
        r'sudah (terbukti|diakui) (WHO|BPOM|FDA) (menyembuhkan|mengobati)',
    ]

    # Pola saran medis langsung yang dilarang
    MEDICAL_ADVICE_TRIGGERS = [
        r'(sebaiknya|harus|wajib) (minum|konsumsi|gunakan) .{0,30} (untuk mengobati)',
        r'dosis (yang tepat|yang benar) untuk (mengobati|menyembuhkan)',
        r'(hentikan|stop) (obat|terapi) dokter',
        r'(lebih baik|lebih aman) dari (obat|terapi) konvensional',
    ]

    # Klaim regulasi yang tidak boleh dibuat
    REGULATORY_VIOLATIONS = [
        r'(disetujui|approved) (FDA|EMA) untuk (mengobati|treatment)',
        r'fitofarmaka (menyembuhkan|mengobati) [a-z]+',
        r'izin edar .{0,20} (terbukti|untuk mengobati)',
    ]

    def check(self, response: str, nav_result: Dict, role: UserRole) -> Tuple[GatekeeperFlag, str, bool]:
        """
        Pemeriksaan multi-lapis.
        Returns: (flag, reason, should_reject)
        """
        response_lower = response.lower()

        # CHECK 1: No-Hallucination
        for pattern in self.HALLUCINATION_TRIGGERS:
            if re.search(pattern, response_lower):
                return (
                    GatekeeperFlag.HALLUCINATION_RISK,
                    f"Terdeteksi klaim berlebihan: '{pattern}'",
                    True
                )

        # CHECK 2: Medical Advice (non-medis role)
        if role not in [UserRole.MEDIS, UserRole.PENELITI]:
            for pattern in self.MEDICAL_ADVICE_TRIGGERS:
                if re.search(pattern, response_lower):
                    return (
                        GatekeeperFlag.MEDICAL_ADVICE,
                        "Terdeteksi saran medis langsung untuk user non-medis",
                        True
                    )

        # CHECK 3: Data tidak ada tapi LLM tetap menjawab
        if not nav_result["data_found"] and len(response) > 100:
            return (
                GatekeeperFlag.HALLUCINATION_RISK,
                "LLM memberikan jawaban panjang tanpa dukungan data database",
                True
            )

        # CHECK 4: Regulatory Violations
        for pattern in self.REGULATORY_VIOLATIONS:
            if re.search(pattern, response_lower):
                return (
                    GatekeeperFlag.POLICY_VIOLATION,
                    f"Pelanggaran klaim regulasi terdeteksi",
                    True
                )

        # LOLOS SEMUA CHECK
        return GatekeeperFlag.CLEAR, "Respons memenuhi semua standar kepatuhan", False

    def generate_rejection(self, flag: GatekeeperFlag, reason: str) -> str:
        """Buat pesan penolakan standar berdasarkan flag"""
        rejection_templates = {
            GatekeeperFlag.HALLUCINATION_RISK: (
                "⚠️ **RESPONS TIDAK DAPAT DITAMPILKAN**\n\n"
                "Sistem mendeteksi potensi informasi yang tidak dapat dikonfirmasi oleh database. "
                "HERBALPHARMA-AGI berkomitmen pada No-Hallucination Policy.\n\n"
                "📋 **DATA TIDAK TERSEDIA DALAM DATABASE** untuk klaim ini.\n\n"
                "Silakan ajukan pertanyaan yang lebih spesifik atau konsultasikan dengan "
                "tenaga farmasi/dokter yang berpengalaman."
            ),
            GatekeeperFlag.MEDICAL_ADVICE: (
                "⚠️ **REKOMENDASI TIDAK DAPAT DITAMPILKAN**\n\n"
                "Sistem mendeteksi konten yang termasuk kategori saran medis langsung. "
                "HERBALPHARMA-AGI bukan pengganti konsultasi medis profesional.\n\n"
                "🩺 Untuk keputusan terapeutik, silakan konsultasikan dengan dokter atau apoteker Anda.\n\n"
                "Informasi yang tersedia dalam database dapat diakses melalui pertanyaan tentang "
                "manfaat umum, cara pengolahan, atau keamanan tanaman herbal."
            ),
            GatekeeperFlag.POLICY_VIOLATION: (
                "⚠️ **KONTEN MELANGGAR KEBIJAKAN REGULASI**\n\n"
                "Respons mengandung klaim yang tidak sesuai dengan regulasi BPOM/WHO yang berlaku. "
                "Sistem menolak untuk menampilkan informasi yang dapat menyesatkan.\n\n"
                "📋 Status regulasi produk herbal dapat diverifikasi melalui situs resmi BPOM: "
                "https://cekbpom.pom.go.id"
            ),
            GatekeeperFlag.NO_DATA: HerbalDatabase.STANDARD_REJECTION,
        }
        return rejection_templates.get(flag, HerbalDatabase.STANDARD_REJECTION)

class LLMBrain:
    """
    Brain: Llama 3.1 8B sebagai Orkestrator utama.
    Menghasilkan respons berdasarkan konteks terstruktur dari D-Syn.
    """

    SYSTEM_PROMPT = """Kamu adalah HERBALPHARMA-AGI, asisten AI farmasi herbal profesional Indonesia.

IDENTITAS:
- Pharmaceutical Scientist (Farmakologi, Farmakognosi, Fitokimia)
- Toxicologist & Clinical Evidence Reviewer
- Regulatory Affairs Specialist (BPOM, WHO, EMA)

ATURAN MUTLAK:
1. HANYA jawab berdasarkan data yang tersedia dalam konteks [DATA TERVERIFIKASI DARI DATABASE]
2. JANGAN mengarang khasiat, senyawa, atau penelitian yang tidak ada di data
3. JANGAN memberikan saran medis langsung (diagnosa, resep, penggantian obat)
4. Jika data tidak tersedia → nyatakan "DATA TIDAK TERSEDIA DALAM DATABASE"
5. Selalu sertakan catatan keamanan yang relevan
6. Gunakan bahasa Indonesia yang profesional dan sesuai target audience
7. Kutip referensi ilmiah yang tersedia di data
8. Setiap klaim harus dapat ditelusuri ke database

DILARANG:
- Mengarang efek terapeutik
- Menyimpulkan klinis tanpa data RCT
- Klaim "menyembuhkan" penyakit
- Merekomendasikan dosis tanpa konfirmasi dokter (kecuali mode Medis/Peneliti)

FORMAT JAWABAN:
- Gunakan format yang sesuai dengan role user
- Sertakan ⚠️ untuk peringatan keamanan penting
- Gunakan 🔬 untuk informasi ilmiah
- Gunakan 📋 untuk referensi"""

    def __init__(self, config: AgentConfig):
        self.config = config
        self.model = None
        self.tokenizer = None
        self._loaded = False

    def load_model(self):
        """Load model Llama 3.1 dengan Unsloth"""
        if self._loaded:
            return

        print(f"⏳ Loading model: {self.config.model_name}")
        try:
            from unsloth import FastLanguageModel

            self.model, self.tokenizer = FastLanguageModel.from_pretrained(
                model_name=self.config.model_name,
                max_seq_length=self.config.max_seq_length,
                dtype=None,
                load_in_4bit=True,
            )
            FastLanguageModel.for_inference(self.model)
            self._loaded = True
            print("✅ Model berhasil di-load!")

        except ImportError:
            print("⚠️ Unsloth tidak tersedia. Menggunakan mode simulasi.")
            self._loaded = True  # Mock mode

        except Exception as e:
            print(f"❌ Error loading model: {e}")
            print("⚠️ Beralih ke mode tanpa LLM (template-based response)")
            self._loaded = True

    def generate(self, context: str, role: UserRole, plant_name: str = None) -> str:
        """
        Generate respons dari LLM atau fallback ke template.
        """
        if not self._loaded:
            self.load_model()

        if self.model is None:
            # Fallback: gunakan data terstruktur dari context langsung
            return self._template_response(context, role, plant_name)

        try:
            prompt = self._build_prompt(context)

            temperature = {
                UserRole.PELAJAR:  0.7,
                UserRole.UMUM:     0.65,
                UserRole.MEDIS:    0.3,
                UserRole.PENELITI: 0.25,
            }.get(role, 0.5)

            inputs = self.tokenizer([prompt], return_tensors="pt").to(self.config.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.config.max_new_tokens,
                    temperature=temperature,
                    do_sample=True,
                    top_p=0.9,
                    repetition_penalty=1.1,
                )

            decoded = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Ekstrak bagian response
            if "### Response:" in decoded:
                response = decoded.split("### Response:")[-1].strip()
            elif "<|start_header_id|>assistant<|end_header_id|>" in decoded:
                response = decoded.split("<|start_header_id|>assistant<|end_header_id|>")[-1].strip()
            else:
                response = decoded

            # Bersihkan token khusus
            response = response.replace("<|eot_id|>", "").replace("<|end_of_text|>", "").strip()
            return response

        except Exception as e:
            print(f"[LLM ERROR] {e}")
            return self._template_response(context, role, plant_name)

    def _build_prompt(self, context: str) -> str:
        """Build prompt dengan format Llama 3.1 Instruct"""
        return (
            f"<|begin_of_text|>"
            f"<|start_header_id|>system<|end_header_id|>\n{self.SYSTEM_PROMPT}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n{context}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n"
        )

    def _template_response(self, context: str, role: UserRole, plant_name: str = None) -> str:
        """
        Template-based fallback saat model tidak tersedia.
        Mengekstrak informasi kunci dari context dan memformat dengan baik.
        """
        # Parse data dari context string
        lines = context.split('\n')
        sections = {}
        current_section = None

        for line in lines:
            if line.startswith('[') and line.endswith(']'):
                current_section = line[1:-1]
                sections[current_section] = []
            elif current_section and line.strip():
                sections[current_section].append(line.strip())

        response_parts = []

        # Header
        if plant_name:
            response_parts.append(f"# 🌿 {plant_name.title()}\n")

        # Sesuaikan kedalaman berdasarkan role
        if role == UserRole.PELAJAR:
            response_parts.append("## 📚 Informasi Umum")
            if "INFORMASI TANAMAN" in sections:
                for line in sections["INFORMASI TANAMAN"][:4]:
                    response_parts.append(line)

            response_parts.append("\n## 🌱 Manfaat Tanaman")
            if "EFEK FARMAKOLOGI TERVERIFIKASI" in sections:
                for eff in sections["EFEK FARMAKOLOGI TERVERIFIKASI"][:3]:
                    parts = eff.split("|")
                    if parts:
                        response_parts.append(f"- {parts[0].replace('•', '').strip()}")

            response_parts.append("\n## ⚠️ Catatan Penting")
            if "KEAMANAN & TOKSIKOLOGI" in sections:
                for line in sections["KEAMANAN & TOKSIKOLOGI"][:2]:
                    if 'Kontraindikasi' in line or 'Catatan' in line:
                        response_parts.append(f"- {line}")

        elif role == UserRole.UMUM:
            response_parts.append("## 🌿 Manfaat")
            if "EFEK FARMAKOLOGI TERVERIFIKASI" in sections:
                for eff in sections["EFEK FARMAKOLOGI TERVERIFIKASI"][:4]:
                    parts = eff.split("|")
                    response_parts.append(f"- **{parts[0].replace('•', '').strip()}**"
                                        + (f": Bukti {parts[1].strip()}" if len(parts) > 1 else ""))

            response_parts.append("\n## 🍵 Cara Pengolahan")
            if "CARA PENGOLAHAN/PENGGUNAAN" in sections:
                for cp in sections["CARA PENGOLAHAN/PENGGUNAAN"][:2]:
                    response_parts.append(f"- {cp.replace('•', '').strip()}")

            response_parts.append("\n## ⚠️ Keamanan & Peringatan")
            if "KEAMANAN & TOKSIKOLOGI" in sections:
                for line in sections["KEAMANAN & TOKSIKOLOGI"][:4]:
                    if any(k in line for k in ['Kontraindikasi', 'Efek Samping', 'Dosis', 'Catatan']):
                        response_parts.append(f"- {line}")
            response_parts.append("\n> 💡 Konsultasikan dengan dokter atau apoteker untuk kondisi medis tertentu.")

        elif role == UserRole.MEDIS:
            response_parts.append("## 🔬 Profil Farmakologi")
            if "KANDUNGAN SENYAWA AKTIF" in sections:
                for s in sections["KANDUNGAN SENYAWA AKTIF"][:3]:
                    response_parts.append(f"- {s.replace('•', '').strip()}")

            response_parts.append("\n## 📊 Efek Klinis (Berdasarkan Level Evidens)")
            if "EFEK FARMAKOLOGI TERVERIFIKASI" in sections:
                for eff in sections["EFEK FARMAKOLOGI TERVERIFIKASI"]:
                    response_parts.append(f"- {eff.replace('•', '').strip()}")

            response_parts.append("\n## ⚠️ Interaksi Obat")
            if "INTERAKSI OBAT - PENTING" in sections:
                for intr in sections["INTERAKSI OBAT - PENTING"]:
                    response_parts.append(f"- {intr}")
            else:
                response_parts.append("- Tidak ada data interaksi signifikan dalam database.")

            response_parts.append("\n## 🏥 Status Regulasi")
            if "STATUS REGULASI" in sections:
                for reg in sections["STATUS REGULASI"]:
                    response_parts.append(f"- {reg.replace('•', '').strip()}")

        else:  # PENELITI
            response_parts.append("## 🔬 Profil Fitokimia Komprehensif")
            for section_name, section_data in sections.items():
                if section_name not in ['INSTRUKSI', 'INSTRUKSI AKHIR']:
                    response_parts.append(f"\n### {section_name}")
                    for line in section_data:
                        response_parts.append(f"- {line.replace('•', '').strip()}")

        response_parts.append(
            "\n---\n"
            "_Semua informasi di atas bersumber dari database HERBALPHARMA-AGI yang terverifikasi. "
            "Untuk informasi lebih lanjut, rujuk referensi ilmiah yang tercantum._"
        )

        return "\n".join(response_parts)



In [7]:
class HerbalPharmaAGI:
    """
    Orkestrator utama HERBALPHARMA-AGI.
    Mengintegrasikan semua komponen dalam pipeline agentic.
    """

    BANNER = """
╔══════════════════════════════════════════════════════════════════════╗
║    🌿 HERBALPHARMA-AGI v1.0 — Agentic Pharmaceutical AI 🌿         ║
║    Farmakognosi & Fitokimia Herbal Indonesia                         ║
╠══════════════════════════════════════════════════════════════════════╣
║  MODEL   : Llama 3.1 8B Instruct (Unsloth 4-bit Quantized)         ║
║  DATABASE: herbal_db.sqlite (SQLite — Source of Truth)              ║
║  POLICY  : No-Hallucination | Regulatory-First | Evidence-Based     ║
╚══════════════════════════════════════════════════════════════════════╝
"""

    WELCOME_MSG = """
Selamat datang di HERBALPHARMA-AGI!

Sebelum memulai, sistem perlu mengenali profil Anda untuk memberikan
informasi yang paling sesuai dan aman.

📌 Silakan pilih peran Anda:
   1. Pelajar      → Informasi edukatif tingkat SMA
   2. Umum         → Manfaat, cara pengolahan, keamanan herbal
   3. Tenaga Medis/Farmasi → Data klinis & farmakokinetik lengkap
   4. Peneliti     → Data ilmiah komprehensif untuk riset

Atau ketikkan pertanyaan langsung — sistem akan mendeteksi peran Anda secara otomatis.

Ketik 'help' untuk bantuan | 'feedback' untuk memberikan penilaian | 'quit' untuk keluar
"""

    def __init__(self, config: AgentConfig = None):
        self.config = config or AgentConfig()
        self.session = SessionState()

        print(self.BANNER)
        print("⏳ Inisialisasi komponen sistem...")

        # Inisialisasi komponen
        self.db = HerbalDatabase(self.config.db_path, self.config.sql_schema_path)
        self.role_detector = RoleDetector()
        self.gnav = GNavAgent(self.db)
        self.dsyn = DSynAgent()
        self.gatekeeper = FinalGatekeeper()
        self.brain = LLMBrain(self.config)

        print("✅ Database berhasil diinisialisasi")
        print("✅ Agen G-Nav dan D-Syn siap")
        print("✅ Final Gatekeeper aktif")
        print("⏳ Loading LLM Brain...")
        self.brain.load_model()

    # ─── PIPELINE UTAMA ─────────────────────────────────────

    def process_query(self, query: str) -> QueryResult:
        """
        Pipeline agentic lengkap:
        Input → Role Check → G-Nav → D-Syn → LLM → Gatekeeper → Output
        """
        start_time = time.time()
        self.session.turn_count += 1

        # ── Step 1: Deteksi / Konfirmasi Role ──────────────────
        if not self.session.role_locked:
            detected_role, confidence = self.role_detector.detect_role(
                query, self.session.user_role
            )
            if confidence > 0.4 or self.session.user_role == UserRole.UNKNOWN:
                if detected_role != self.session.user_role:
                    self.session.user_role = detected_role

        role = self.session.user_role
        if role == UserRole.UNKNOWN:
            role = UserRole.UMUM

        # ── Step 2: G-Nav — Navigasi Graf Pengetahuan ──────────
        nav_result = self.gnav.navigate(query, role)
        intent = nav_result["intent"]
        plant_name = nav_result.get("plant_name")

        # ── Step 3: Cek Data Tersedia ──────────────────────────
        if not nav_result["data_found"]:
            processing_time = time.time() - start_time
            rejection = HerbalDatabase.STANDARD_REJECTION
            self._log(query, intent, nav_result["sql_log"],
                      ResponseType.REJECTED, GatekeeperFlag.NO_DATA)

            return QueryResult(
                query=query, intent=intent,
                sql_executed=nav_result["sql_log"],
                rag_context="",
                raw_data={},
                llm_response=rejection,
                gatekeeper_flag=GatekeeperFlag.NO_DATA,
                response_type=ResponseType.REJECTED,
                sources=[],
                confidence=0.0,
                processing_time=processing_time,
            )

        # ── Step 4: D-Syn — Sintesis Konteks ──────────────────
        context = self.dsyn.synthesize_context(nav_result, role, query)

        # ── Step 5: LLM Brain — Generate Respons ──────────────
        raw_response = self.brain.generate(context, role, plant_name)

        # ── Step 6: Final Gatekeeper — Compliance Check ────────
        flag, reason, should_reject = self.gatekeeper.check(raw_response, nav_result, role)

        if should_reject:
            final_response = self.gatekeeper.generate_rejection(flag, reason)
            resp_type = ResponseType.REJECTED
        else:
            final_response = raw_response
            resp_type = ResponseType.ANSWERED

        # ── Step 7: Audit Log ───────────────────────────────────
        processing_time = time.time() - start_time
        self._log(query, intent, nav_result["sql_log"], resp_type, flag)

        # Simpan ke history sesi
        self.session.history.append({
            "turn": self.session.turn_count,
            "role": role.value,
            "query": query,
            "response": final_response[:200] + "..." if len(final_response) > 200 else final_response,
            "flag": flag.value,
            "time": f"{processing_time:.2f}s"
        })

        return QueryResult(
            query=query,
            intent=intent,
            sql_executed=nav_result["sql_log"],
            rag_context=context[:500],
            raw_data=nav_result["data"],
            llm_response=final_response,
            gatekeeper_flag=flag,
            response_type=resp_type,
            sources=nav_result["sources"],
            confidence=nav_result["confidence"],
            processing_time=processing_time,
        )

    # ─── DISPLAY FUNCTIONS ──────────────────────────────────

    def display_result(self, result: QueryResult):
        """Tampilkan hasil dengan format yang rapi"""
        role = self.session.user_role

        role_colors = {
            UserRole.PELAJAR:  "🎓",
            UserRole.UMUM:     "👤",
            UserRole.MEDIS:    "🩺",
            UserRole.PENELITI: "🔬",
        }
        role_emoji = role_colors.get(role, "🤖")

        print("\n" + "═" * 70)
        print(f"{role_emoji} [{role.value.upper()}] — {result.intent.replace('_', ' ').title()}")
        print(f"⏱️  {result.processing_time:.2f}s | 🔍 SQL: {len(result.sql_executed)} queries")
        print("═" * 70)

        if result.gatekeeper_flag != GatekeeperFlag.CLEAR:
            print(f"🛡️  GATEKEEPER: {result.gatekeeper_flag.value}")
            print()

        print(result.llm_response)

        if result.sources:
            print("\n📚 Sumber:")
            for src in result.sources[:3]:
                if src.strip():
                    print(f"   • {src}")

        print("\n" + "─" * 70)
        print("Apakah jawaban ini bermanfaat? Ketik 'feedback' untuk memberikan penilaian.")

    def handle_special_commands(self, user_input: str) -> bool:
        """Handle perintah khusus. Returns True jika dihandle."""
        cmd = user_input.strip().lower()

        if cmd in ['1', 'pelajar']:
            self.session.user_role = UserRole.PELAJAR
            self.session.role_locked = True
            print("\n" + self.role_detector.format_role_greeting(UserRole.PELAJAR))
            return True

        elif cmd in ['2', 'umum']:
            self.session.user_role = UserRole.UMUM
            self.session.role_locked = True
            print("\n" + self.role_detector.format_role_greeting(UserRole.UMUM))
            return True

        elif cmd in ['3', 'medis', 'tenaga medis', 'farmasi']:
            self.session.user_role = UserRole.MEDIS
            self.session.role_locked = True
            print("\n" + self.role_detector.format_role_greeting(UserRole.MEDIS))
            return True

        elif cmd in ['4', 'peneliti', 'riset', 'industri']:
            self.session.user_role = UserRole.PENELITI
            self.session.role_locked = True
            print("\n" + self.role_detector.format_role_greeting(UserRole.PENELITI))
            return True

        elif cmd == 'help':
            self._show_help()
            return True

        elif cmd == 'status':
            self._show_status()
            return True

        elif cmd.startswith('feedback'):
            self._handle_feedback()
            return True

        elif cmd == 'history':
            self._show_history()
            return True

        elif cmd == 'plants' or cmd == 'daftar tanaman':
            self._show_available_plants()
            return True

        elif cmd in ['quit', 'exit', 'keluar']:
            print("\n👋 Terima kasih telah menggunakan HERBALPHARMA-AGI.")
            print(f"   Session ID: {self.session.session_id}")
            print(f"   Total pertanyaan: {self.session.turn_count}")
            return True

        return False

    def _show_help(self):
        help_text = """
╔═══════════════════════════════════════════════════════════╗
║               HERBALPHARMA-AGI — BANTUAN                 ║
╠═══════════════════════════════════════════════════════════╣
║  PERINTAH KHUSUS:                                        ║
║  1 / pelajar    → Mode Pelajar SMA                       ║
║  2 / umum       → Mode Masyarakat Umum                   ║
║  3 / medis      → Mode Tenaga Medis/Farmasi              ║
║  4 / peneliti   → Mode Peneliti/Industri                 ║
║  status         → Lihat status sesi                      ║
║  plants         → Daftar tanaman dalam database          ║
║  history        → Riwayat percakapan                     ║
║  feedback       → Beri penilaian jawaban                 ║
║  quit/exit      → Keluar dari sesi                       ║
╠═══════════════════════════════════════════════════════════╣
║  CONTOH PERTANYAAN:                                      ║
║  • "Apa manfaat kunyit?"                                 ║
║  • "Cara merebus jahe yang benar?"                       ║
║  • "Interaksi obat sambiloto dengan warfarin?"           ║
║  • "Farmakokinetik andrografolid?"                       ║
║  • "Tanaman herbal untuk antiinflamasi?"                 ║
║  • "Status regulasi BPOM pegagan?"                       ║
╚═══════════════════════════════════════════════════════════╝
"""
        print(help_text)

    def _show_status(self):
        print(f"""
📊 STATUS SESI HERBALPHARMA-AGI
═══════════════════════════════════
Session ID    : {self.session.session_id}
Mode Aktif    : {self.session.user_role.value.upper()}
Role Terkunci : {'Ya' if self.session.role_locked else 'Tidak (auto-detect)'}
Total Tanya   : {self.session.turn_count}
Feedback      : {len(self.session.feedback_log)} entri
═══════════════════════════════════
""")

    def _show_history(self):
        print("\n📜 RIWAYAT PERCAKAPAN:")
        print("═" * 60)
        if not self.session.history:
            print("Belum ada riwayat percakapan.")
        for entry in self.session.history[-5:]:
            print(f"[{entry['turn']}] [{entry['role']}] Q: {entry['query'][:60]}...")
            print(f"     ⏱️ {entry['time']} | Flag: {entry['flag']}")
        print("═" * 60)

    def _show_available_plants(self):
        rows, _ = self.db.execute_query(
            "SELECT nama_lokal, nama_ilmiah, familia, status_bpom FROM tanaman ORDER BY nama_lokal"
        )
        print("\n🌿 TANAMAN DALAM DATABASE HERBALPHARMA-AGI:")
        print("═" * 65)
        print(f"{'No':<4} {'Nama Lokal':<18} {'Nama Ilmiah':<28} {'Status BPOM'}")
        print("─" * 65)
        for i, r in enumerate(rows, 1):
            print(f"{i:<4} {r['nama_lokal']:<18} {r['nama_ilmiah']:<28} {r['status_bpom']}")
        print("═" * 65)
        print(f"Total: {len(rows)} tanaman terverifikasi")

    def _handle_feedback(self):
        print("\n📝 FEEDBACK — Bantu kami meningkatkan kualitas jawaban:")
        if not self.session.history:
            print("Belum ada jawaban untuk dinilai.")
            return

        last = self.session.history[-1]
        print(f"Pertanyaan terakhir: {last['query']}")

        try:
            rating_str = input("Rating (1-5, 1=Buruk, 5=Sangat Baik): ").strip()
            rating = int(rating_str)
            if not 1 <= rating <= 5:
                raise ValueError

            feedback_text = input("Komentar (opsional, tekan Enter untuk skip): ").strip()

            self.db.log_feedback(
                session_id=self.session.session_id,
                query=last['query'],
                response=last['response'],
                rating=rating,
                feedback=feedback_text,
                role=last['role']
            )

            self.session.feedback_log.append({
                "query": last['query'],
                "rating": rating,
                "feedback": feedback_text
            })

            print(f"✅ Terima kasih atas feedback Anda! Rating: {'⭐' * rating}")

        except (ValueError, KeyboardInterrupt):
            print("Feedback dibatalkan.")

    def _log(self, query: str, intent: str, sql_log: List[str],
             resp_type: ResponseType, flag: GatekeeperFlag):
        if self.config.audit_enabled:
            self.db.log_audit(
                session_id=self.session.session_id,
                user_role=self.session.user_role.value,
                query=query,
                intent=intent,
                sql="; ".join(sql_log),
                resp_type=resp_type.value,
                flag=flag.value
            )

    # ─── MAIN INTERACTIVE LOOP ──────────────────────────────

    def run(self):
        """Jalankan sesi interaktif HERBALPHARMA-AGI"""
        print(self.WELCOME_MSG)

        while True:
            try:
                user_input = input(f"\n[{self.session.user_role.value.upper()}] Anda: ").strip()

                if not user_input:
                    continue

                # Handle perintah khusus
                if self.handle_special_commands(user_input):
                    if user_input.lower() in ['quit', 'exit', 'keluar']:
                        break
                    continue

                # Process query biasa
                print("\n⏳ Memproses pertanyaan Anda...")
                result = self.process_query(user_input)
                self.display_result(result)

            except KeyboardInterrupt:
                print("\n\n👋 Sesi diakhiri. Terima kasih menggunakan HERBALPHARMA-AGI!")
                break
            except Exception as e:
                print(f"\n❌ Error: {e}")
                print("Silakan coba lagi atau ketik 'help' untuk bantuan.")


def run_demo(agi: HerbalPharmaAGI):
    """
    Demo otomatis dengan berbagai skenario untuk setiap role.
    """
    print("\n" + "═" * 70)
    print("🧪 DEMO HERBALPHARMA-AGI — Testing Semua Role & Skenario")
    print("═" * 70)

    demo_cases = [
        # (role, query, description)
        (UserRole.PELAJAR,   "Apa itu kunyit dan mengapa warnanya kuning?",
         "Pelajar: Pertanyaan dasar tentang kunyit"),

        (UserRole.UMUM,      "Bagaimana cara membuat minuman jahe yang benar dan berapa takarannya?",
         "Umum: Cara pengolahan jahe"),

        (UserRole.UMUM,      "Apakah sambiloto aman untuk ibu hamil?",
         "Umum: Keamanan sambiloto kehamilan"),

        (UserRole.MEDIS,     "Berapa dosis andrografolid untuk infeksi saluran napas atas dan apa interaksinya dengan antihipertensi?",
         "Medis: Dosis klinis & interaksi obat sambiloto"),

        (UserRole.MEDIS,     "Jelaskan mekanisme aksi kurkumin sebagai antiinflamasi berdasarkan studi klinis.",
         "Medis: Mekanisme aksi berbasis evidens"),

        (UserRole.PENELITI,  "Berikan profil fitokimia lengkap Curcuma longa termasuk jalur metabolit, farmakokinetik, dan referensi untuk pengembangan fitofarmaka.",
         "Peneliti: Profil komprehensif untuk riset"),

        (UserRole.PENELITI,  "Tanaman herbal apa yang terbukti memiliki aktivitas antioksidan dengan level bukti A?",
         "Peneliti: Query berbasis efek dengan filter level bukti"),

        (UserRole.UMUM,      "Apakah mengkudu bisa menyembuhkan kanker secara 100%?",
         "Umum: Query berpotensi hallucination — harus ditolak Gatekeeper"),

        (UserRole.UMUM,      "Informasi tentang bunga tulip herbal untuk diabetes",
         "Umum: Tanaman tidak ada di database — harus ditolak"),
    ]

    for i, (role, query, desc) in enumerate(demo_cases, 1):
        print(f"\n{'─' * 70}")
        print(f"📋 Demo #{i}: {desc}")
        print(f"   Role  : {role.value.upper()}")
        print(f"   Query : {query}")
        print(f"{'─' * 70}")

        # Set role
        agi.session.user_role = role
        agi.session.role_locked = True

        result = agi.process_query(query)
        agi.display_result(result)

        print(f"\n   ✅ Gatekeeper: {result.gatekeeper_flag.value}")
        print(f"   ✅ Response Type: {result.response_type.value}")
        print(f"   ✅ SQL Queries: {result.sql_executed}")

        time.sleep(0.5)  # Jeda antar demo

    print("\n" + "═" * 70)
    print("✅ Demo selesai! Semua skenario berhasil ditest.")

    # Tampilkan audit log
    print("\n📊 AUDIT LOG (5 Terakhir):")
    logs, _ = agi.db.execute_query(
        "SELECT * FROM audit_log ORDER BY id DESC LIMIT 5"
    )
    for log in logs:
        print(f"   [{log['timestamp']}] [{log['user_role'].upper()}] "
              f"Intent:{log['intent_detected']} | Type:{log['response_type']} | "
              f"Flag:{log['gatekeeper_flag']}")



In [8]:
def main():
    """
    Entry point utama HERBALPHARMA-AGI.
    Pilih mode: demo otomatis atau sesi interaktif.
    """
    print("=" * 70)
    print("HERBALPHARMA-AGI — Mode Startup")
    print("=" * 70)
    print("1. Demo Otomatis (test semua skenario)")
    print("2. Mode Interaktif (chat langsung)")
    print("=" * 70)

    # Inisialisasi sistem
    agi = HerbalPharmaAGI(config=AgentConfig(
        model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
        sql_schema_path="herbal_db.sql",
        db_path="herbal_db.sqlite",
        max_new_tokens=512,
        audit_enabled=True,
    ))

    try:
        choice = input("\nPilih mode (1/2) atau tekan Enter untuk interaktif: ").strip()
    except (EOFError, KeyboardInterrupt):
        choice = "2"

    if choice == "1":
        run_demo(agi)
        print("\n\n💬 Lanjut ke mode interaktif? (y/n): ", end="")
        try:
            cont = input().strip().lower()
            if cont == 'y':
                agi.session.role_locked = False
                agi.session.user_role = UserRole.UNKNOWN
                agi.run()
        except (EOFError, KeyboardInterrupt):
            pass
    else:
        agi.run()

if __name__ == "__main__":
    main()

# ============================================================
# CELL 14: QUICK-START — Untuk Colab (jalankan tanpa main())
# ============================================================
# Uncomment baris berikut untuk langsung jalankan demo di Colab:

# agi = HerbalPharmaAGI()
# run_demo(agi)

# Atau untuk mode interaktif:
# agi = HerbalPharmaAGI()
# agi.run()

# Atau untuk query tunggal:
# agi = HerbalPharmaAGI()
# agi.session.user_role = UserRole.MEDIS
# result = agi.process_query("Jelaskan farmakokinetik kurkumin dan interaksi obatnya")
# agi.display_result(result)


# ============================================================
# ╔══════════════════════════════════════════════════════════╗
# ║     MODUL FINE-TUNING — HERBALPHARMA-AGI v2.0           ║
# ║     LoRA Fine-Tuning + Evaluasi + Export GGUF           ║
# ╚══════════════════════════════════════════════════════════╝
# ============================================================



HERBALPHARMA-AGI — Mode Startup
1. Demo Otomatis (test semua skenario)
2. Mode Interaktif (chat langsung)

╔══════════════════════════════════════════════════════════════════════╗
║    🌿 HERBALPHARMA-AGI v1.0 — Agentic Pharmaceutical AI 🌿         ║
║    Farmakognosi & Fitokimia Herbal Indonesia                         ║
╠══════════════════════════════════════════════════════════════════════╣
║  MODEL   : Llama 3.1 8B Instruct (Unsloth 4-bit Quantized)         ║
║  DATABASE: herbal_db.sqlite (SQLite — Source of Truth)              ║
║  POLICY  : No-Hallucination | Regulatory-First | Evidence-Based     ║
╚══════════════════════════════════════════════════════════════════════╝

⏳ Inisialisasi komponen sistem...
✅ Database berhasil diinisialisasi
✅ Agen G-Nav dan D-Syn siap
✅ Final Gatekeeper aktif
⏳ Loading LLM Brain...
⏳ Loading model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everythi

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

✅ Model berhasil di-load!

Pilih mode (1/2) atau tekan Enter untuk interaktif: 1

══════════════════════════════════════════════════════════════════════
🧪 DEMO HERBALPHARMA-AGI — Testing Semua Role & Skenario
══════════════════════════════════════════════════════════════════════

──────────────────────────────────────────────────────────────────────
📋 Demo #1: Pelajar: Pertanyaan dasar tentang kunyit
   Role  : PELAJAR
   Query : Apa itu kunyit dan mengapa warnanya kuning?
──────────────────────────────────────────────────────────────────────
[DB ERROR] no such table: tanaman
[DB WRITE ERROR] no such table: audit_log

══════════════════════════════════════════════════════════════════════
🎓 [PELAJAR] — Cari Tanaman
⏱️  0.00s | 🔍 SQL: 1 queries
══════════════════════════════════════════════════════════════════════
🛡️  GATEKEEPER: NO_DATA

⚠️ DATA TIDAK TERSEDIA DALAM DATABASE.

Sistem hanya menjawab berdasarkan data terverifikasi dalam basis data HERBALPHARMA-AGI. Informasi yang Anda tan

In [ ]:
# MODE 1: Demo otomatis (rekomendasikan untuk pertama kali)
agi = HerbalPharmaAGI()
run_demo(agi)

# ─────────────────────────────────────────────────
# MODE 2: Chat interaktif (uncomment untuk pakai)
# agi = HerbalPharmaAGI()
# agi.run()

# ─────────────────────────────────────────────────
# MODE 3: Query tunggal langsung
# agi = HerbalPharmaAGI()
# agi.session.user_role = UserRole.PENELITI
# result = agi.process_query('Profil fitokimia Curcuma longa untuk pengembangan fitofarmaka')
# agi.display_result(result)


In [9]:
"""
Memuat model Llama 3.1 8B base untuk fine-tuning.
Jalankan SEKALI di awal sesi sebelum training atau inference.
"""

from google.colab import userdata
from unsloth import FastLanguageModel
import torch

# ── Hyperparameter Global ──────────────────────────────────
max_seq_length = 2048  # RoPE Scaling otomatis didukung
dtype          = None  # None = auto (Float16 T4, BFloat16 Ampere+)
load_in_4bit   = True  # 4-bit quantization hemat VRAM

# Model base untuk fine-tuning HERBALPHARMA-AGI
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

print(f"⏳ Loading base model: {model_name}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = model_name,
    max_seq_length = max_seq_length,
    dtype          = dtype,
    load_in_4bit   = load_in_4bit,
    # token        = userdata.get('HF_TOKEN'),  # Uncomment jika model private
)
print("✅ Base model berhasil di-load!")

"""
Konfigurasi LoRA (Low-Rank Adaptation) untuk fine-tuning
efisien pada model besar tanpa mengubah semua parameter.
Khusus dikonfigurasi untuk domain farmasi herbal.
"""

model = FastLanguageModel.get_peft_model(
    model,
    r                   = 16,        # Rank LoRA: 8/16/32/64/128 — lebih besar = lebih ekspresif
    target_modules      = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha          = 16,        # Scaling factor LoRA (biasanya = r)
    lora_dropout        = 0,         # 0 = dioptimasi oleh Unsloth
    bias                = "none",    # "none" dioptimasi
    use_gradient_checkpointing = "unsloth",  # 30% hemat VRAM, support context panjang
    random_state        = 3407,
    use_rslora          = False,     # Rank Stabilized LoRA — opsional
    loftq_config        = None,      # LoftQ quantization — opsional
)

print("✅ LoRA adapter dikonfigurasi!")
print(f"   Target modules : q/k/v/o_proj, gate/up/down_proj")
print(f"   LoRA rank (r)  : 16")
print(f"   LoRA alpha     : 16")
print(f"   Gradient ckpt  : unsloth (VRAM-efficient)")



⏳ Loading base model: unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit
==((====))==  Unsloth 2026.3.3: Fast Llama patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Base model berhasil di-load!


Unsloth 2026.3.3 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


✅ LoRA adapter dikonfigurasi!
   Target modules : q/k/v/o_proj, gate/up/down_proj
   LoRA rank (r)  : 16
   LoRA alpha     : 16
   Gradient ckpt  : unsloth (VRAM-efficient)


In [10]:
"""
Format dataset untuk fine-tuning HERBALPHARMA-AGI.
Dataset harus berformat JSONL dengan field:
  - instruction: Instruksi tugas (termasuk role context)
  - input: Pertanyaan/konteks user
  - output: Jawaban ideal berdasarkan database

Format Alpaca digunakan karena kompatibel dengan Llama 3.1 Instruct.
"""

from datasets import load_dataset

# ── Template Prompt Alpaca (WAJIB konsisten antara train & inference) ──
HERBALPHARMA_PROMPT = """Below is an instruction that describes a pharmaceutical query task for HERBALPHARMA-AGI system. Respond professionally based only on verified database information.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token  # WAJIB: Tanpa ini, generasi tidak berhenti

def formatting_prompts_func(examples):
    """
    Formatter untuk dataset training HERBALPHARMA-AGI.
    Menggabungkan instruction + input + output ke format Alpaca.
    """
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts        = []

    for instruction, input_text, output in zip(instructions, inputs, outputs):
        # Format lengkap dengan EOS token
        text = HERBALPHARMA_PROMPT.format(instruction, input_text, output) + EOS_TOKEN
        texts.append(text)

    return {"text": texts}


# ── Load Dataset ────────────────────────────────────────────
# PETUNJUK: Ganti path file sesuai dataset HERBALPHARMA Anda
# Format yang didukung: JSONL, JSON, CSV
# Field wajib: instruction, input, output

# Contoh load dari file lokal:
TRAIN_DATASET_PATH = "herbalpharma_dataset_train.jsonl"  # ← Ganti sesuai file Anda
TEST_DATASET_PATH  = "herbalpharma_dataset_test.jsonl"   # ← Ganti sesuai file Anda

# ── Buat Dataset Dummy jika file tidak ada (untuk testing) ─
import os, json as _json

def create_dummy_dataset(path: str, n: int = 20, split: str = "train"):
    """Buat dataset dummy untuk testing pipeline."""
    if os.path.exists(path):
        return
    samples = []
    role_samples = {
        "pelajar": [
            ("Jelaskan sebagai pelajar SMA tentang tanaman obat berikut.",
             "Apa itu kunyit?",
             "Kunyit (Curcuma longa) adalah tanaman rimpang berwarna kuning-oranye dari keluarga Zingiberaceae. Warna kuningnya berasal dari senyawa kurkumin. Tanaman ini banyak digunakan sebagai bumbu masak dan obat tradisional karena memiliki sifat anti-peradangan dan antioksidan."),
        ],
        "umum": [
            ("Berikan informasi umum yang praktis untuk masyarakat awam.",
             "Bagaimana cara membuat rebusan jahe yang benar?",
             "Cara membuat rebusan jahe: 1) Siapkan 2-3 cm rimpang jahe segar. 2) Cuci bersih dan geprek. 3) Rebus dengan 300 ml air hingga mendidih selama 10 menit. 4) Saring dan minum selagi hangat. Konsumsi 2-3x sehari setelah makan. ⚠️ Batasi jika Anda memiliki gangguan asam lambung."),
        ],
        "medis": [
            ("Berikan informasi klinis untuk tenaga medis dan farmasi.",
             "Sebutkan interaksi obat sambiloto dengan antihipertensi.",
             "Andrographis paniculata memiliki efek hipotensif melalui mekanisme vasodilatasi. Kombinasi dengan antihipertensi konvensional dapat menyebabkan hipotensi aditif (Level Bukti B). Rekomendasi klinis: monitor tekanan darah secara berkala. Referensi: Burgos et al., 2009, J Ethnopharmacol."),
        ],
        "peneliti": [
            ("Berikan data ilmiah komprehensif untuk keperluan riset.",
             "Jelaskan jalur metabolisme kurkumin dan bioavailabilitasnya.",
             "Kurkumin dimetabolisme via CYP1A2 dan CYP3A4 (inhibitor lemah). Bioavailabilitas oral < 1% karena metabolisme first-pass ekstensif. Metabolit utama: tetrahidrokurkumin dan heksahidrokurkumin. T½: 6-8 jam. Protein binding: 65%. Ko-administrasi piperin 20 mg meningkatkan bioavailabilitas 20x (Anand et al., 2007, Mol Pharm)."),
        ],
    }
    for i in range(n):
        role = list(role_samples.keys())[i % 4]
        instr, inp, out = role_samples[role][0]
        samples.append({"instruction": instr, "input": inp, "output": out, "role": role})
    with open(path, 'w') as f:
        for s in samples:
            f.write(_json.dumps(s, ensure_ascii=False) + "\n")
    print(f"   ✅ Dataset dummy dibuat: {path} ({n} samples)")

# Buat dummy jika belum ada
create_dummy_dataset(TRAIN_DATASET_PATH, n=40)
create_dummy_dataset(TEST_DATASET_PATH, n=10, split="test")

# ── Load & Format Dataset ───────────────────────────────────
dataset_train = load_dataset("json", data_files=TRAIN_DATASET_PATH, split="train")
dataset_train = dataset_train.map(formatting_prompts_func, batched=True)

dataset_test  = load_dataset("json", data_files=TEST_DATASET_PATH, split="train")
dataset_test  = dataset_test.map(formatting_prompts_func, batched=True)

print(f"\n✅ Dataset berhasil di-load!")
print(f"   Train samples : {len(dataset_train)}")
print(f"   Test samples  : {len(dataset_test)}")
print(f"\n📋 Contoh data TRAIN:")
print(dataset_train[0]["text"][:500] + "...")
print(f"\n📋 Contoh data TEST:")
print(dataset_test[0]["text"][:500] + "...")



   ✅ Dataset dummy dibuat: herbalpharma_dataset_train.jsonl (40 samples)
   ✅ Dataset dummy dibuat: herbalpharma_dataset_test.jsonl (10 samples)


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]


✅ Dataset berhasil di-load!
   Train samples : 40
   Test samples  : 10

📋 Contoh data TRAIN:
Below is an instruction that describes a pharmaceutical query task for HERBALPHARMA-AGI system. Respond professionally based only on verified database information.

### Instruction:
Jelaskan sebagai pelajar SMA tentang tanaman obat berikut.

### Input:
Apa itu kunyit?

### Response:
Kunyit (Curcuma longa) adalah tanaman rimpang berwarna kuning-oranye dari keluarga Zingiberaceae. Warna kuningnya berasal dari senyawa kurkumin. Tanaman ini banyak digunakan sebagai bumbu masak dan obat tradisional k...

📋 Contoh data TEST:
Below is an instruction that describes a pharmaceutical query task for HERBALPHARMA-AGI system. Respond professionally based only on verified database information.

### Instruction:
Jelaskan sebagai pelajar SMA tentang tanaman obat berikut.

### Input:
Apa itu kunyit?

### Response:
Kunyit (Curcuma longa) adalah tanaman rimpang berwarna kuning-oranye dari keluarga Zingiberaceae

In [11]:
"""
Konfigurasi SFTTrainer (Supervised Fine-Tuning Trainer) dari TRL.
Hyperparameter dioptimasi untuk fine-tuning domain farmasi herbal
dengan GPU T4/A100 di Google Colab.
"""

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

# ── Cek Memory GPU sebelum Training ───────────────────────
gpu_stats          = torch.cuda.get_device_properties(0)
start_gpu_memory   = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory         = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)

print(f"📊 GPU Status Sebelum Training:")
print(f"   GPU Name       : {gpu_stats.name}")
print(f"   Total Memory   : {max_memory} GB")
print(f"   Memory Reserved: {start_gpu_memory} GB")
print(f"   Memory Free    : {round(max_memory - start_gpu_memory, 3)} GB\n")

# ── SFTTrainer Setup ────────────────────────────────────────
trainer = SFTTrainer(
    model               = model,
    tokenizer           = tokenizer,
    train_dataset       = dataset_train,
    dataset_text_field  = "text",
    max_seq_length      = max_seq_length,
    dataset_num_proc    = 2,
    packing             = False,   # True = 5x lebih cepat untuk sequence pendek

    args = TrainingArguments(
        # ── Batch & Gradient ──────────────────────────────
        per_device_train_batch_size  = 1,
        gradient_accumulation_steps  = 8,    # Efektif batch size = 8
        # ── Scheduler & Warmup ───────────────────────────
        warmup_steps                 = 50,
        num_train_epochs             = 2,    # Tambah epoch untuk dataset kecil
        # ── Learning Rate ────────────────────────────────
        learning_rate                = 2e-4,
        lr_scheduler_type            = "linear",
        # ── Presisi & Optimasi ────────────────────────────
        fp16                         = not is_bfloat16_supported(),
        bf16                         = is_bfloat16_supported(),
        optim                        = "adamw_8bit",
        weight_decay                 = 0.01,
        # ── Logging & Output ─────────────────────────────
        logging_steps                = 20,
        output_dir                   = "outputs_herbalpharma",
        report_to                    = "none",   # Ganti "wandb" jika pakai W&B
        # ── Reproducibility ──────────────────────────────
        seed                         = 3407,
    ),
)

print("✅ Trainer berhasil dikonfigurasi!")
print(f"   Epochs         : 2")
print(f"   Batch size     : 1 (effective: 8 via grad accum)")
print(f"   Learning rate  : 2e-4")
print(f"   Optimizer      : AdamW 8-bit")
print(f"   Precision      : {'BF16' if is_bfloat16_supported() else 'FP16'}")
print(f"   Output dir     : outputs_herbalpharma/")



📊 GPU Status Sebelum Training:
   GPU Name       : Tesla T4
   Total Memory   : 14.563 GB
   Memory Reserved: 10.834 GB
   Memory Free    : 3.729 GB



Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/40 [00:00<?, ? examples/s]

✅ Trainer berhasil dikonfigurasi!
   Epochs         : 2
   Batch size     : 1 (effective: 8 via grad accum)
   Learning rate  : 2e-4
   Optimizer      : AdamW 8-bit
   Precision      : FP16
   Output dir     : outputs_herbalpharma/


In [12]:
"""
Jalankan fine-tuning HERBALPHARMA-AGI.
Estimasi waktu: ~15-30 menit pada T4 untuk dataset kecil.
Pantau loss di log setiap 20 langkah.
"""

print("🚀 Memulai fine-tuning HERBALPHARMA-AGI...")
print("   Tips: Loss yang baik untuk domain medis: < 1.0 di akhir epoch\n")

trainer_stats = trainer.train()

print("\n✅ Training selesai!")

"""
Laporan penggunaan memori GPU dan durasi training.
"""

used_memory          = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage      = round(used_memory / max_memory * 100, 3)
lora_percentage      = round(used_memory_for_lora / max_memory * 100, 3)

print("\n" + "="*55)
print("📊 LAPORAN TRAINING HERBALPHARMA-AGI")
print("="*55)
print(f"Durasi Training         : {trainer_stats.metrics['train_runtime']:.1f} detik")
print(f"                          ({round(trainer_stats.metrics['train_runtime']/60, 2)} menit)")
print(f"Train Loss Akhir        : {trainer_stats.metrics.get('train_loss', 'N/A'):.4f}")
print(f"Steps Per Second        : {trainer_stats.metrics.get('train_steps_per_second', 0):.2f}")
print(f"Samples Per Second      : {trainer_stats.metrics.get('train_samples_per_second', 0):.2f}")
print(f"─"*55)
print(f"Peak Memory Reserved    : {used_memory} GB")
print(f"Memory untuk LoRA       : {used_memory_for_lora} GB")
print(f"% Penggunaan GPU        : {used_percentage} %")
print(f"% LoRA dari GPU         : {lora_percentage} %")
print("="*55)



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 40 | Num Epochs = 2 | Total steps = 10
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


🚀 Memulai fine-tuning HERBALPHARMA-AGI...
   Tips: Loss yang baik untuk domain medis: < 1.0 di akhir epoch



Step,Training Loss



✅ Training selesai!

📊 LAPORAN TRAINING HERBALPHARMA-AGI
Durasi Training         : 74.4 detik
                          (1.24 menit)
Train Loss Akhir        : 2.6018
Steps Per Second        : 0.13
Samples Per Second      : 1.07
───────────────────────────────────────────────────────
Peak Memory Reserved    : 12.377 GB
Memory untuk LoRA       : 1.543 GB
% Penggunaan GPU        : 84.989 %
% LoRA dari GPU         : 10.595 %


In [13]:
"""
Evaluasi komprehensif model HERBALPHARMA-AGI setelah fine-tuning.
Metrik:
  - ROUGE-L   : Kesamaan frasa (untuk TIME mode & respons terstruktur)
  - BERTScore : Kualitas semantik (untuk GENERAL mode)
  - Logic Acc : Kepatuhan format khusus (untuk SENSOR mode)
  - Latency   : Kecepatan inferensi rata-rata
"""

import numpy as np
from tqdm import tqdm
from rouge_score import rouge_scorer as rs_module
from bert_score import score as bertscore

# ── Aktifkan Mode Inferensi ────────────────────────────────
FastLanguageModel.for_inference(model)

def detect_eval_mode(instruction: str, input_text: str) -> tuple:
    """
    Deteksi mode evaluasi dari instruksi.
    Returns (mode_name, temperature)
    """
    instr_lower = instruction.lower()
    # Mode khusus domain HERBALPHARMA
    if any(kw in instr_lower for kw in ["medis", "klinis", "dosis", "farmakokinetik"]):
        return "CLINICAL", 0.3   # Presisi tinggi, low temperature
    elif any(kw in instr_lower for kw in ["peneliti", "riset", "fitokimia", "jalur"]):
        return "RESEARCH", 0.25  # Sangat presisi untuk riset
    elif any(kw in instr_lower for kw in ["pelajar", "siswa", "mahasiswa"]):
        return "STUDENT", 0.7    # Lebih kreatif untuk penjelasan
    else:
        return "GENERAL", 0.65   # Mode umum

def generate_response(instruction: str, input_text: str,
                       temperature: float = 0.65) -> str:
    """Generate respons model untuk evaluasi."""
    prompt = HERBALPHARMA_PROMPT.format(instruction, input_text, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens     = 256,
            temperature        = temperature,
            do_sample          = True,
            top_p              = 0.9,
            repetition_penalty = 1.1,
        )

    decoded   = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred_text = decoded.split("### Response:")[-1].strip()
    pred_text = pred_text.replace("<|end_of_text|>", "").strip()
    return pred_text

# ── Container Metrik ────────────────────────────────────────
eval_stats = {
    "total": 0, "gen_time": [],
    "format_ok": 0, "format_total": 0,
}

preds_by_mode = {"CLINICAL": [], "RESEARCH": [], "STUDENT": [], "GENERAL": []}
refs_by_mode  = {"CLINICAL": [], "RESEARCH": [], "STUDENT": [], "GENERAL": []}

rouge_scorer_obj = rs_module.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# ── Load Test Dataset ───────────────────────────────────────
test_samples = []
with open(TEST_DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            test_samples.append(json.loads(line))

print(f"🚀 Memulai Evaluasi HERBALPHARMA-AGI pada {len(test_samples)} data...\n")

for sample in tqdm(test_samples, desc="📊 Evaluasi"):
    eval_stats["total"] += 1

    mode, temp = detect_eval_mode(sample['instruction'], sample['input'])

    start = time.time()
    pred_text = generate_response(sample['instruction'], sample['input'], temp)
    eval_stats["gen_time"].append(time.time() - start)

    gt_text = sample["output"].strip()

    # ── Cek Format Respons (Kepatuhan Data-Driven) ─────────
    # Respons yang baik harus mengandung data factual dari database
    factual_indicators = [
        "N/A", "Referensi:", "Level Bukti", "Dosis", "⚠️",
        "BPOM", "WHO", "mg", "kontraindikasi"
    ]
    has_factual = any(ind.lower() in pred_text.lower() for ind in factual_indicators)
    if has_factual or mode in ["STUDENT", "GENERAL"]:
        eval_stats["format_ok"] += 1
    eval_stats["format_total"] += 1

    preds_by_mode[mode].append(pred_text)
    refs_by_mode[mode].append(gt_text)

# ── Cetak Hasil Evaluasi ─────────────────────────────────────
print("\n" + "="*55)
print("📊 FINAL EVALUATION RESULTS — HERBALPHARMA-AGI")
print("="*55)
print(f"Total Sampel Uji      : {eval_stats['total']}")
print(f"Avg Latency           : {np.mean(eval_stats['gen_time']):.3f}s / respons")
print(f"Min/Max Latency       : {min(eval_stats['gen_time']):.3f}s / {max(eval_stats['gen_time']):.3f}s")

# Format Score
if eval_stats["format_total"] > 0:
    fmt_acc = eval_stats["format_ok"] / eval_stats["format_total"]
    print(f"\n[RESPONSE QUALITY - FORMAT SCORE]")
    print(f"Accuracy              : {fmt_acc:.2%} ({eval_stats['format_ok']}/{eval_stats['format_total']})")
    print(f"Target                : > 80% (respons mengandung data faktual)")

# ROUGE per mode
print(f"\n[ROUGE-L SCORE — PER MODE]")
all_preds, all_refs = [], []
for mode_name in ["CLINICAL", "RESEARCH", "STUDENT", "GENERAL"]:
    p_list = preds_by_mode[mode_name]
    r_list = refs_by_mode[mode_name]
    if p_list:
        r_scores = [
            rouge_scorer_obj.score(ref, pred)['rougeL'].fmeasure
            for ref, pred in zip(r_list, p_list)
        ]
        print(f"  {mode_name:<12} : ROUGE-L = {np.mean(r_scores):.4f}  (n={len(p_list)})")
        all_preds.extend(p_list)
        all_refs.extend(r_list)

if all_preds:
    overall_rouge = [
        rouge_scorer_obj.score(r, p)['rougeL'].fmeasure
        for r, p in zip(all_refs, all_preds)
    ]
    print(f"  {'OVERALL':<12} : ROUGE-L = {np.mean(overall_rouge):.4f}")
    print(f"  Target       : > 0.35 (Baik), > 0.50 (Sangat Baik)")

# BERTScore keseluruhan
if all_preds:
    print(f"\n[BERTScore — KUALITAS SEMANTIK]")
    try:
        P, R, F1 = bertscore(all_preds, all_refs, lang="id", verbose=False)
        print(f"  Precision (P) : {P.mean().item():.4f}")
        print(f"  Recall (R)    : {R.mean().item():.4f}")
        print(f"  F1 Score      : {F1.mean().item():.4f}")
        print(f"  Target        : F1 > 0.70 (Baik), > 0.80 (Sangat Baik)")
    except Exception as e:
        print(f"  ⚠️  BERTScore gagal (kemungkinan OOM): {e}")

print("="*55)



🚀 Memulai Evaluasi HERBALPHARMA-AGI pada 10 data...



📊 Evaluasi: 100%|██████████| 10/10 [03:56<00:00, 23.69s/it]



📊 FINAL EVALUATION RESULTS — HERBALPHARMA-AGI
Total Sampel Uji      : 10
Avg Latency           : 23.693s / respons
Min/Max Latency       : 14.326s / 41.602s

[RESPONSE QUALITY - FORMAT SCORE]
Accuracy              : 70.00% (7/10)
Target                : > 80% (respons mengandung data faktual)

[ROUGE-L SCORE — PER MODE]
  CLINICAL     : ROUGE-L = 0.1081  (n=2)
  RESEARCH     : ROUGE-L = 0.1078  (n=2)
  STUDENT      : ROUGE-L = 0.1795  (n=3)
  GENERAL      : ROUGE-L = 0.2156  (n=3)
  OVERALL      : ROUGE-L = 0.1617
  Target       : > 0.35 (Baik), > 0.50 (Sangat Baik)

[BERTScore — KUALITAS SEMANTIK]


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

  Precision (P) : 0.6626
  Recall (R)    : 0.7236
  F1 Score      : 0.6916
  Target        : F1 > 0.70 (Baik), > 0.80 (Sangat Baik)


In [ ]:
"""
Simpan LoRA adapter ke lokal dan upload ke Hugging Face Hub.
LoRA adapter jauh lebih kecil (~100-300MB) dibanding model full.
"""

from huggingface_hub import HfApi, create_repo, update_repo_visibility

# ── Konfigurasi ─────────────────────────────────────────────
HF_USERNAME        = "YOUR_HF_USERNAME"          # ← Ganti dengan username HF Anda
ADAPTER_REPO_NAME  = "herbalpharma-agi-v1-adapter"
ADAPTER_REPO_ID    = f"{HF_USERNAME}/{ADAPTER_REPO_NAME}"
ADAPTER_LOCAL_DIR  = "herbalpharma_lora_adapter"

print(f"💾 Menyimpan LoRA Adapter ke lokal: ./{ADAPTER_LOCAL_DIR}/")
model.save_pretrained(ADAPTER_LOCAL_DIR)
tokenizer.save_pretrained(ADAPTER_LOCAL_DIR)
print(f"✅ Adapter tersimpan lokal!\n")

# ── Upload ke Hugging Face ──────────────────────────────────
print(f"⏳ Mengupload LoRA Adapter ke HuggingFace: {ADAPTER_REPO_ID}")
print("   (Estimasi: 1-3 menit)\n")

try:
    my_token = userdata.get('HF_TOKEN')  # Ambil dari Colab Secrets
    api = HfApi()

    create_repo(
        repo_id   = ADAPTER_REPO_ID,
        repo_type = "model",
        exist_ok  = True,
        token     = my_token,
        private   = True,
    )

    # Upload menggunakan push_to_hub Unsloth
    model.push_to_hub(ADAPTER_REPO_ID, token=my_token)
    tokenizer.push_to_hub(ADAPTER_REPO_ID, token=my_token)

    print("="*55)
    print("✅ LoRA ADAPTER BERHASIL DIUPLOAD!")
    print(f"   Link : https://huggingface.co/{ADAPTER_REPO_ID}")
    print("="*55)

except Exception as e:
    print(f"❌ Gagal upload adapter: {e}")
    print("   ➡️  Pastikan HF_TOKEN sudah di-set di Colab Secrets (🔑 menu kiri)")
    print(f"   ➡️  Adapter tersimpan lokal di: ./{ADAPTER_LOCAL_DIR}/")



In [ ]:
"""
Konversi model ke format GGUF untuk deployment lokal
menggunakan Ollama, LM Studio, llama.cpp, dll.

Q5_K_M = Quantisasi 5-bit K-Means Mixed:
  - Kualitas: ~99% dari model float16
  - Ukuran: ~5-6 GB (dari ~16GB)
  - Cocok untuk: production deployment, CPU inference
"""

GGUF_LOCAL_DIR   = "herbalpharma_gguf_q5"
GGUF_REPO_NAME   = "herbalpharma-agi-v1-gguf"
GGUF_REPO_ID     = f"{HF_USERNAME}/{GGUF_REPO_NAME}"
GGUF_FILENAME    = "herbalpharma-agi-v1.Q5_K_M.gguf"

print("🔄 Mengkonversi model ke format GGUF Q5_K_M...")
print("   Kualitas : High Quality (~99% dari FP16)")
print("   Estimasi : 3-8 menit...\n")

model.save_pretrained_gguf(
    GGUF_LOCAL_DIR,
    tokenizer,
    quantization_method = "q5_k_m"
)

print(f"✅ GGUF Q5_K_M berhasil disimpan ke: ./{GGUF_LOCAL_DIR}/")

# Cek ukuran file yang dihasilkan
import glob
gguf_files = glob.glob(f"{GGUF_LOCAL_DIR}/*.gguf")
if gguf_files:
    for gf in gguf_files:
        size_gb = os.path.getsize(gf) / (1024**3)
        print(f"   📁 {os.path.basename(gf)} — {size_gb:.2f} GB")


# ── Upload GGUF ke HuggingFace ──────────────────────────────
print(f"\n⏳ Mengupload GGUF ke HuggingFace: {GGUF_REPO_ID}")
print("   ⚠️  GGUF besar (~5-6 GB). Estimasi: 5-15 menit.\n")
print("   Jangan tutup tab Colab!\n")

# Cari file GGUF yang dihasilkan
gguf_source_files = glob.glob(f"{GGUF_LOCAL_DIR}/*.gguf")
if not gguf_source_files:
    print("❌ File GGUF tidak ditemukan. Periksa folder output.")
else:
    gguf_source_path = gguf_source_files[0]  # Ambil file pertama
    actual_filename  = os.path.basename(gguf_source_path)

    try:
        my_token = userdata.get('HF_TOKEN')
        api = HfApi()

        # Buat repo GGUF (terpisah dari adapter)
        create_repo(
            repo_id   = GGUF_REPO_ID,
            repo_type = "model",
            exist_ok  = True,
            token     = my_token,
            private   = True,
        )

        # Upload file GGUF
        api.upload_file(
            path_or_fileobj = gguf_source_path,
            path_in_repo    = GGUF_FILENAME,
            repo_id         = GGUF_REPO_ID,
            repo_type       = "model",
            token           = my_token,
        )

        # Buat README otomatis untuk repo GGUF
        readme_content = f"""---
license: llama3
base_model: {model_name}
tags:
  - herbalpharma
  - pharmaceutical
  - indonesian
  - herbal
  - gguf
  - unsloth
  - llama-3
language:
  - id
  - en
---

# HERBALPHARMA-AGI v1.0 — GGUF Q5_K_M

Agentic Pharmaceutical AI untuk Farmakognosi & Fitokimia Herbal Indonesia.

## Model Info
- **Base Model**: {model_name}
- **Fine-tuning**: LoRA (r=16, alpha=16)
- **Quantization**: Q5_K_M (~99% quality)
- **Domain**: Farmasi Herbal Indonesia (BPOM/WHO compliant)

## Usage (Ollama)
```bash
ollama run {GGUF_REPO_ID}
```

## Disclaimer
Model ini hanya untuk tujuan edukasi dan penelitian.
Bukan pengganti konsultasi tenaga medis profesional.
"""
        api.upload_file(
            path_or_fileobj = readme_content.encode(),
            path_in_repo    = "README.md",
            repo_id         = GGUF_REPO_ID,
            repo_type       = "model",
            token           = my_token,
        )

        print("="*55)
        print("✅ GGUF BERHASIL DIUPLOAD KE HUGGING FACE!")
        print(f"   Repo  : https://huggingface.co/{GGUF_REPO_ID}")
        print(f"   File  : {GGUF_FILENAME}")
        print("="*55)

    except Exception as e:
        print(f"❌ Gagal upload GGUF: {e}")
        print(f"   ➡️  File tersimpan lokal: ./{gguf_source_path}")
        print("   ➡️  Cek HF_TOKEN dan koneksi internet")



In [ ]:

print("\n" + "╔" + "═"*53 + "╗")
print("║     ✅ HERBALPHARMA-AGI PIPELINE SELESAI!           ║")
print("╠" + "═"*53 + "╣")
print("║  OUTPUT YANG DIHASILKAN:                            ║")
print(f"║  📁 LoRA Adapter  : ./{ADAPTER_LOCAL_DIR:<30}║")
print(f"║  📁 GGUF Q5_K_M   : ./{GGUF_LOCAL_DIR:<30}║")
print(f"║  📁 Training Logs : ./outputs_herbalpharma/        ║")
print(f"║  🗄️  Database      : ./herbal_db.sqlite             ║")
print("╠" + "═"*53 + "╣")
print("║  HUGGING FACE:                                      ║")
print(f"║  🔗 Adapter : hf.co/{ADAPTER_REPO_ID[:32]:<32}║")
print(f"║  🔗 GGUF    : hf.co/{GGUF_REPO_ID[:32]:<32}║")
print("╠" + "═"*53 + "╣")
print("║  LANGKAH SELANJUTNYA:                               ║")
print("║  1. Download GGUF untuk Ollama/LM Studio           ║")
print("║  2. Jalankan AGI system: HerbalPharmaAGI().run()  ║")
print("║  3. Tambah data ke herbal_db.sql untuk ekspansi   ║")
print("╚" + "═"*53 + "╝")

